# 3. SPRINT 2 — Preparación de datos y modelado inicial

**Proyecto de investigación:** *Modelo predictivo de horas de indisponibilidad operativa mediante Gradient Boosting Machine (GBM) considerando variables meteorológicas y oceanográficas en el Terminal Portuario ENGIE Perú, Ilo.*

**Marco metodológico:** CRISP-DM (fase *Preparación de datos*) · **Marco de gestión:** Scrum · **Sprint:** Sprint 2 — Semanas 7 a 11

**Estructura de esta parte del notebook**

| N.° | Historia de usuario / Tarea | Contenido |
|---|---|---|
| 3.1 | **HU05** — Limpieza del dataset | |
| 3.1.1 | T09 | Tratar valores nulos y duplicados (**5 story points**) |
| 3.1.2 | T10 | Corregir inconsistencias en horas de indisponibilidad (**3 story points**) |

> **Insumos (salidas del Sprint 1):** `dataset_integrado_horario_HU03.csv` (malla horaria integrada) y `Historico_Indisponibilidad_Operativa*.csv` (eventos con `fecha_inicio` / `fecha_fin`). Opcionalmente, los CSV crudos del datalogger, para re-verificar duplicados por marca de tiempo.
>
> **Salidas de HU05 (insumo de HU06 – Ingeniería de características):** dataset horario limpio + banderas de imputación + eventos depurados + bitácoras de decisiones.

## 3.1 HU05 — Limpieza del dataset

**Historia de usuario:** *Como Científico de datos, quiero depurar los registros históricos, para mejorar la calidad de los datos utilizados por el modelo.*

### Criterios de aceptación de HU05 (TB1)

1. Se aplica y documenta el tratamiento de **nulos, duplicados e inconsistencias**.
2. Se alcanza una **completitud ≥ 90 %** o se **justifica** el tratamiento de los registros faltantes.
3. Los vacíos se tratan **de forma diferenciada por variable** (la tolerancia a la imputación no es igual para viento que para oleaje).
4. Cada decisión (imputar, excluir, corregir) queda **trazable** en una bitácora.

### Regla de tratamiento de faltantes (inciso f del pipeline)

| Tipo de vacío | Causa típica | Tratamiento |
|---|---|---|
| **Puntual** (≤ límite de horas de la variable) | falla breve de sensor | **Interpolación temporal** (lineal o spline), acotada al máximo de horas continuas |
| **Prolongado** (> límite) | mantenimiento o falla de estación | **No se imputa**: se documenta y se evalúa excluir el bloque temporal o la variable |

Antes de remuestrear/agregar por hora, se verifica y **deduplica por marca de tiempo**, conservando la **última lectura válida**. En el Sprint 1 este paso se ejecutó dentro del remuestreo y no detectó duplicados en ninguna de las ocho series; aquí se **re-verifica de forma explícita y documentada**.

In [ ]:
# ==== HU05 - Preparación del entorno (estilo y funciones de dibujo compartidas) ====
import glob, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 50)

COLOR_NAVY    = '#1B3A5C'
COLOR_TEAL    = '#2A9D8F'
COLOR_TEAL_LT = '#D6EFEC'
COLOR_ORANGE  = '#E9762B'
COLOR_RED     = '#C43D3D'
COLOR_GRAY    = '#9AA5B1'
COLOR_GRAY_LT = '#EEF1F4'
COLOR_GOLD    = '#E3A857'
plt.rcParams['font.family'] = 'DejaVu Sans'

def caja(ax, x, y, w, h, texto, color=COLOR_NAVY, texto_color='white', fontsize=10, fontweight='bold'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.05',
                                linewidth=1.5, edgecolor=color, facecolor=color, zorder=2))
    ax.text(x + w/2, y + h/2, texto, ha='center', va='center', fontsize=fontsize,
            color=texto_color, fontweight=fontweight, zorder=3, linespacing=1.35)

def flecha(ax, p1, p2, color=COLOR_NAVY, lw=1.8, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=15, color=color,
                                 lw=lw, connectionstyle=f'arc3,rad={rad}', zorder=1))

def lienzo(w=11, h=5, xlim=(0, 11), ylim=(0, 5)):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis('off')
    return fig, ax

MPH_A_MS = 0.44704
print('Entorno listo.')

### Vista rápida: el flujo completo de HU05

```
T09 (nulos y duplicados) ─────────────────────────────►  T10 (inconsistencias en horas)
Serie horaria: dedup → rango físico → vacíos → imputar    Eventos: nulos → duplicados → fechas → duración
```

In [ ]:
fig, ax = lienzo(13, 5.2, xlim=(0, 13), ylim=(0, 5.4))
ax.set_title('HU05 — Limpieza del dataset: de los insumos del Sprint 1 al dataset entrenable',
             fontsize=13, color=COLOR_NAVY, fontweight='bold', pad=12)

caja(ax, 0.2, 3.6, 2.3, 1.1, 'Entrada\nDataset horario HU03\n+ histórico de eventos', color=COLOR_GRAY, fontsize=8.8)
flecha(ax, (2.5, 4.15), (3.0, 4.15))

# Carril T09
ax.text(3.0, 5.15, 'T09 · Nulos y duplicados (series horarias)', fontsize=9.5, color=COLOR_TEAL, fontweight='bold')
pasos9 = ['1. Verificar y\ndeduplicar por\nmarca de tiempo', '2. Rango físico\n(valores imposibles\n→ nulo)', '3. Diagnóstico\nde vacíos:\npuntual vs prolongado',
          '4. Interpolación\nlineal / spline /\ncircular acotada']
for i, p in enumerate(pasos9):
    x = 3.0 + i * 2.5
    caja(ax, x, 3.6, 2.2, 1.1, p, color=COLOR_TEAL, fontsize=8.2)
    if i < 3: flecha(ax, (x + 2.2, 4.15), (x + 2.5, 4.15), color=COLOR_TEAL)

# Carril T10
ax.text(3.0, 2.85, 'T10 · Inconsistencias en horas de indisponibilidad (eventos)', fontsize=9.5, color=COLOR_ORANGE, fontweight='bold')
pasos10 = ['5. Nulos y\nduplicados\nen eventos', '6. Fechas\ninvertidas y\nduración ≠ fin−inicio', '7. Formato, solapes\ny causa no\nmeteo-oceanográfica']
for i, p in enumerate(pasos10):
    x = 3.0 + i * 2.5
    caja(ax, x, 1.4, 2.2, 1.1, p, color=COLOR_ORANGE, fontsize=8.2)
    if i < 2: flecha(ax, (x + 2.2, 1.95), (x + 2.5, 1.95), color=COLOR_ORANGE)

flecha(ax, (11.6, 3.6), (11.6, 2.5), color=COLOR_TEAL)
caja(ax, 10.4, 1.4, 2.4, 1.1, 'Targets rehechos\ncon eventos\nlimpios', color=COLOR_NAVY, fontsize=8.5)
flecha(ax, (8.0, 1.95), (10.4, 1.95), color=COLOR_ORANGE)

flecha(ax, (11.6, 1.4), (11.6, 0.95), color=COLOR_NAVY)
caja(ax, 3.0, 0.15, 9.8, 0.8, 'Salida → HU06: dataset limpio + banderas de imputación + eventos depurados + bitácoras de decisión', color=COLOR_NAVY, fontsize=9.3)
plt.tight_layout(); plt.show()

### Configuración de HU05 (parámetros editables)

Todo lo que es **criterio de negocio** vive en una sola celda, para que pueda **validarse con la asesora y el Product Owner** y re-ejecutarse sin tocar el código.

> **Nota:** los límites de horas (`max_gap_h`) y los rangos físicos son **valores de partida razonables, no resultados medidos**. Se contrastan con el diagnóstico de vacíos (T09.3) y se ajustan si la asesora o Operaciones lo indican.

**Lógica de la tolerancia diferenciada por variable**
- **Viento (velocidad y dirección)** y **oleaje**: cambian rápido (ráfagas, grupos de olas) → límite **corto**, porque interpolar horas largas inventaría condiciones que pudieron ser justo las críticas para la operación.
- **Temperatura, presión y humedad**: variables **suaves** → límite **más amplio**; la temperatura y la presión usan **spline cúbico local** por su evolución continua.
- **Dirección del viento**: se interpola de forma **circular** (componentes seno/coseno), porque interpolar grados directamente falla en el salto 360° → 0°.
- **Diagnóstico del datalogger** (temperatura interna, voltaje): no son predictores; se tratan para conservar la trazabilidad y explicar vacíos compartidos.

In [ ]:
# ==== HU05 - Configuración (EDITABLE) ====
# rol: 'predictor' entra al modelo; 'diagnostico' solo explica fallas del datalogger
# metodo: 'linear' | 'spline' (cúbico local) | 'circular' (grados, vía seno/coseno)
# max_gap_h: máximo de horas CONSECUTIVAS sin dato que se interpolan (vacío puntual)
# rango: valores físicamente posibles; fuera de rango -> se convierte en nulo antes de imputar
# max_pct_vacio: si tras imputar queda más de este % de vacío, se evalúa excluir la variable
CONFIG_VARIABLES = {
    'velocidad_viento_ms':              dict(rol='predictor',   metodo='linear',   max_gap_h=3, rango=(0, 60),    max_pct_vacio=10),
    'direccion_viento_deg':             dict(rol='predictor',   metodo='circular', max_gap_h=3, rango=(0, 360),   max_pct_vacio=10),
    'temperatura_c':                    dict(rol='predictor',   metodo='spline',   max_gap_h=6, rango=(-5, 45),   max_pct_vacio=10),
    'humedad_relativa_pct':             dict(rol='predictor',   metodo='linear',   max_gap_h=6, rango=(0, 100),   max_pct_vacio=10),
    'presion_atmosferica_hpa':          dict(rol='predictor',   metodo='spline',   max_gap_h=6, rango=(900, 1100), max_pct_vacio=10),
    'altura_olas_m':                    dict(rol='predictor',   metodo='linear',   max_gap_h=2, rango=(0, 15),    max_pct_vacio=10),
    'temperatura_interna_datalogger_c': dict(rol='diagnostico', metodo='linear',   max_gap_h=6, rango=(-10, 80),  max_pct_vacio=10),
    'voltaje_datalogger_v':             dict(rol='diagnostico', metodo='linear',   max_gap_h=6, rango=(0, 30),    max_pct_vacio=10),
}
VARIABLES = list(CONFIG_VARIABLES)
PREDICTORES = [v for v, c in CONFIG_VARIABLES.items() if c['rol'] == 'predictor']
DIAGNOSTICO = [v for v, c in CONFIG_VARIABLES.items() if c['rol'] == 'diagnostico']

UMBRAL_COMPLETITUD_PCT = 90          # criterio de aceptación HU05 (TB1)
VENTANA_SPLINE_H = 12                # puntos válidos a cada lado del hueco para el spline local
HORIZONTE_H = 24                     # horizonte de predicción (HU01/T02)

# ---- T10: reglas sobre el histórico de eventos ----
TOL_DURACION_H = 0.05                # tolerancia entre duración registrada y (fin - inicio)
UMBRAL_REVISION_H = 1.0              # diferencias mayores a esto se marcan para validar con Operaciones
POLITICA_FECHA_FIN_NULA = 'imputar_con_duracion_registrada'  # o 'excluir'. Imputa solo si duracion_h_registrada y duracion_h_calculada coinciden
FUSIONAR_SOLAPADOS = False           # False: solo marca solapes; True: fusiona eventos solapados
COLUMNAS_CAUSA_CANDIDATAS = ['causa_registrada', 'causa', 'causa_indisponibilidad', 'motivo', 'tipo_causa']
PALABRAS_CAUSA_NO_CLIMATICA = ['mantenim', 'falla', 'electric', 'energ', 'averia', 'mecanic', 'programad', 'comunicacion', 'corte de']

display(pd.DataFrame(CONFIG_VARIABLES).T.rename(columns={
    'rol': 'Rol', 'metodo': 'Método', 'max_gap_h': 'Máx. horas a interpolar',
    'rango': 'Rango físico', 'max_pct_vacio': '% vacío máx. tolerado'}))

### Carga de los insumos del Sprint 1

Se leen `dataset_integrado_horario_HU03.csv` y el histórico de eventos. Si no están en la sesión de Colab, la celda ofrece subirlos. **Las columnas de targets provisionales de HU03 se descartan**: se reconstruirán al final con los eventos ya limpios.

In [ ]:
# ==== Carga de insumos ====
RUTA_DATASET = 'dataset_integrado_horario_HU03.csv'
PATRON_EVENTOS = 'Historico_Indisponibilidad_Operativa*.csv'

faltantes = [p for p in [RUTA_DATASET, PATRON_EVENTOS] if not glob.glob(p)]
if faltantes:
    try:
        from google.colab import files
        print('Faltan archivos; selecciona los CSV a subir:', *faltantes, sep='\n  - ')
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'Faltan archivos en el directorio de trabajo: {faltantes}')

df_hu03 = pd.read_csv(RUTA_DATASET, index_col='fecha_hora', parse_dates=['fecha_hora'])
ruta_eventos = glob.glob(PATRON_EVENTOS)[0]
df_eventos_crudo = pd.read_csv(ruta_eventos, parse_dates=['fecha_inicio', 'fecha_fin'])

# Validación mínima de esquema
faltan_vars = [v for v in VARIABLES if v not in df_hu03.columns]
faltan_cols_ev = [c for c in ['id_evento', 'fecha_inicio', 'fecha_fin'] if c not in df_eventos_crudo.columns]
assert not faltan_vars, f'Faltan variables en el dataset HU03: {faltan_vars}'
assert not faltan_cols_ev, f'Faltan columnas en el histórico de eventos: {faltan_cols_ev}'

print(f'Dataset HU03 : {df_hu03.shape[0]:,} filas x {df_hu03.shape[1]} columnas  ({df_hu03.index.min()} -> {df_hu03.index.max()})')
print(f'Histórico    : {len(df_eventos_crudo):,} eventos  (archivo: {Path(ruta_eventos).name})')
display(df_eventos_crudo.head(3))

## 3.1.1 T09 — Tratar valores nulos y duplicados

### T09.1 Verificación y deduplicación por marca de tiempo (paso previo a la agregación horaria)

Se verifican **tres niveles**:

1. **CSV crudos** (si están en la sesión): marcas de tiempo repetidas por variable. Si una marca se repite se conserva la **última lectura válida** (no nula).
2. **Índice de la malla horaria**: no debe haber horas repetidas ni saltos.
3. **Tabla resultante**: confirma o refuta el hallazgo del Sprint 1 (*0 duplicados en las ocho series*).

In [ ]:
# ==== T09.1 - Verificación y deduplicación por marca de tiempo ====
PATRONES_CRUDOS = {
    'velocidad_viento_mph':              'Velocidad de Viento*.csv',
    'direccion_viento_deg':              'Direccion de Viento*.csv',
    'temperatura_c':                     'Temperatura-*.csv',
    'humedad_relativa_pct':              'Humedad Relativa*.csv',
    'presion_atmosferica_hpa':           'Presion Atmosferica*.csv',
    'altura_olas_m':                     'Altura de Olas*.csv',
    'temperatura_interna_datalogger_c':  'Temperatura Interna de Datalogger*.csv',
    'voltaje_datalogger_v':              'Voltaje de Datalogger*.csv',
}

def _a_numero(v):
    if pd.isna(v): return np.nan
    m = re.search(r'-?\d+(\.\d+)?', str(v).strip().replace(',', '.'))
    return float(m.group()) if m else np.nan

def deduplicar_ultima_lectura_valida(df):
    """Ordena por marca de tiempo y, si una marca se repite, conserva la ULTIMA lectura VALIDA
    (no nula). Si todas las lecturas repetidas son nulas, conserva una fila nula."""
    df = df.dropna(subset=['fecha_hora']).copy()
    df['_orden'] = np.arange(len(df))
    df['_valida'] = df['valor'].notna()
    df = df.sort_values(['fecha_hora', '_valida', '_orden'])      # las válidas quedan al final de cada marca
    return df.drop_duplicates('fecha_hora', keep='last').drop(columns=['_orden', '_valida'])

def verificar_duplicados_crudo(columna, patron):
    rutas = glob.glob(patron)
    if not rutas:
        return {'serie': columna, 'fuente': 'CSV crudo no disponible', 'lecturas': np.nan,
                'marcas_repetidas': np.nan, 'con_valores_distintos': np.nan, 'lecturas_tras_dedup': np.nan}
    df = pd.read_csv(rutas[0], encoding='utf-8')
    df = df.iloc[:, :2]; df.columns = ['Time', 'crudo']
    df['fecha_hora'] = pd.to_datetime(df['Time'], errors='coerce')
    df['valor'] = df['crudo'].apply(_a_numero)
    rep = df.dropna(subset=['fecha_hora']).duplicated('fecha_hora', keep=False)
    n_rep = int(df.dropna(subset=['fecha_hora']).duplicated('fecha_hora').sum())
    n_conf = int(df.dropna(subset=['fecha_hora'])[rep].groupby('fecha_hora')['valor'].nunique(dropna=True).gt(1).sum()) if rep.any() else 0
    dedup = deduplicar_ultima_lectura_valida(df[['fecha_hora', 'valor']])
    return {'serie': columna, 'fuente': Path(rutas[0]).name, 'lecturas': len(df),
            'marcas_repetidas': n_rep, 'con_valores_distintos': n_conf, 'lecturas_tras_dedup': len(dedup)}

reporte_dup_crudo = pd.DataFrame([verificar_duplicados_crudo(c, p) for c, p in PATRONES_CRUDOS.items()])
display(reporte_dup_crudo)

# --- Nivel 2: índice de la malla horaria
indice_duplicado = int(df_hu03.index.duplicated().sum())
df_hu03 = df_hu03[~df_hu03.index.duplicated(keep='last')].sort_index()
malla = pd.date_range(df_hu03.index.min(), df_hu03.index.max(), freq='h')
horas_faltantes_en_malla = len(malla) - len(df_hu03)
df_h = df_hu03.reindex(malla)
df_h.index.name = 'fecha_hora'

duplicados_total_crudo = reporte_dup_crudo['marcas_repetidas'].sum(skipna=True)
resumen_dup = pd.DataFrame([
    ['Series crudas verificadas', int(reporte_dup_crudo['lecturas'].notna().sum()), 'de 8'],
    ['Marcas de tiempo repetidas en CSV crudos', int(duplicados_total_crudo), 'Sprint 1 reportó 0 en las 8 series'],
    ['Horas repetidas en el índice horario', indice_duplicado, 'se conserva la última'],
    ['Horas ausentes en la malla (se reinsertan como nulo)', int(horas_faltantes_en_malla), 'malla horaria continua'],
], columns=['Verificación', 'Resultado', 'Comentario'])
display(resumen_dup)

### T09.2 Valores físicamente imposibles → nulo

Antes de medir vacíos, cualquier valor fuera del **rango físico** configurado se convierte en nulo y **se cuenta**. Es parte del tratamiento de nulos (un valor imposible equivale a una lectura inexistente). **No** se tocan los *outliers* estadísticos (IQR) de T08: son valores posibles, y por ejemplo el 8 % de `voltaje_datalogger_v` es un hallazgo a documentar, no a corregir.

In [ ]:
# ==== T09.2 - Rango físico ====
df_crudo_h = df_h[VARIABLES].copy()          # copia previa a cualquier tratamiento (para el "antes")
df_rango = df_crudo_h.copy()
filas = []
for v, cfg in CONFIG_VARIABLES.items():
    lo, hi = cfg['rango']
    fuera = (df_rango[v] < lo) | (df_rango[v] > hi)
    filas.append({'variable': v, 'rango_fisico': f'[{lo}, {hi}]', 'valores_fuera_de_rango': int(fuera.sum())})
    df_rango.loc[fuera, v] = np.nan
reporte_rango = pd.DataFrame(filas)
display(reporte_rango)

### T09.3 Diagnóstico de vacíos: puntuales vs prolongados

Se construye una **bitácora de vacíos** (una fila por hueco: variable, inicio, fin, horas, posición) y cada hueco se clasifica con el límite **de su propia variable**:

- `puntual`: horas consecutivas ≤ `max_gap_h` → **se interpola**.
- `prolongado`: horas consecutivas > `max_gap_h` → **no se interpola**, se documenta.
- Los huecos en el **borde** de la serie (inicio o final) no tienen dato a ambos lados, por lo que **nunca** se interpolan.

In [ ]:
# ==== T09.3 - Bitácora de vacíos ====
def listar_vacios(serie, max_gap_h):
    es_nulo = serie.isna()
    grupo = (es_nulo != es_nulo.shift(fill_value=False)).cumsum()
    filas = []
    for _, bloque in serie.groupby(grupo):
        if not bloque.isna().all():
            continue
        ini, fin, n = bloque.index[0], bloque.index[-1], len(bloque)
        if ini == serie.index[0]:   pos = 'borde_inicial'
        elif fin == serie.index[-1]: pos = 'borde_final'
        else:                        pos = 'interno'
        if pos != 'interno':         clase = 'prolongado (borde)' if n > max_gap_h else 'puntual (borde, no interpolable)'
        else:                        clase = 'puntual' if n <= max_gap_h else 'prolongado'
        filas.append({'variable': serie.name, 'inicio': ini, 'fin': fin, 'horas': n, 'posicion': pos,
                      'max_gap_h_variable': max_gap_h, 'clasificacion': clase})
    return filas

bitacora_vacios = pd.DataFrame(
    [f for v in VARIABLES for f in listar_vacios(df_rango[v], CONFIG_VARIABLES[v]['max_gap_h'])]
)

if bitacora_vacios.empty:
    print('No se detectaron vacíos en ninguna variable.')
else:
    resumen_vacios = (bitacora_vacios
        .pivot_table(index='variable', columns='clasificacion', values='horas', aggfunc=['count', 'sum'], fill_value=0))
    resumen_vacios.columns = [f'{a}_{b}'.replace('count', 'n_huecos').replace('sum', 'horas') for a, b in resumen_vacios.columns]
    display(resumen_vacios)
    print('Huecos más largos por variable:')
    display(bitacora_vacios.sort_values('horas', ascending=False).groupby('variable').head(2).reset_index(drop=True))

In [ ]:
# ==== Vista visual - Mapa de disponibilidad (antes de imputar) ====
disp = (df_rango[VARIABLES].isna().resample('D').mean() * 100).T       # % de horas vacías por día
fig, ax = plt.subplots(figsize=(14, 4.2))
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('disp', ['white', COLOR_GOLD, COLOR_RED])
from matplotlib.colors import PowerNorm
im = ax.imshow(disp.values, aspect='auto', cmap=cmap, norm=PowerNorm(gamma=0.4, vmin=0, vmax=100),
               extent=[mdates.date2num(disp.columns[0]), mdates.date2num(disp.columns[-1]), len(VARIABLES), 0])
ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
ax.set_yticks(np.arange(len(VARIABLES)) + 0.5); ax.set_yticklabels(VARIABLES, fontsize=8.5)
ax.set_title('T09.3 — ¿Cuándo faltan datos? % de horas vacías por día y variable (antes de imputar)',
             fontsize=12, color=COLOR_NAVY, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.85, label='% de horas vacías en el día (escala realzada)')
fig.tight_layout(); plt.show()
print('Lectura: franjas verticales que cruzan TODAS las filas = falla de estación/datalogger; '
      'marcas aisladas en una fila = falla de un sensor.')

In [ ]:
# ==== Vista visual - ¿Qué tan largos son los huecos vs el límite de cada variable? ====
if not bitacora_vacios.empty:
    fig, axes = plt.subplots(2, 4, figsize=(16, 6.6))
    for ax, v in zip(axes.ravel(), VARIABLES):
        h = bitacora_vacios.loc[bitacora_vacios['variable'] == v, 'horas']
        lim = CONFIG_VARIABLES[v]['max_gap_h']
        if len(h):
            tope = max(h.max(), lim + 1)
            ax.hist(h.clip(upper=48), bins=np.arange(0.5, min(tope, 48) + 1.5, 1),
                    color=[COLOR_TEAL], edgecolor='white')
        ax.axvline(lim + 0.5, color=COLOR_RED, lw=1.5, ls='--', label=f'límite = {lim} h')
        ax.set_title(v, fontsize=9, color=COLOR_NAVY, fontweight='bold')
        ax.set_xlabel('horas consecutivas sin dato (tope 48)', fontsize=8)
        ax.legend(fontsize=7.5, frameon=False); ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('Huecos a la izquierda de la línea roja se interpolan; a la derecha se documentan',
                 fontsize=12.5, color=COLOR_NAVY, fontweight='bold')
    fig.tight_layout(); plt.show()

### T09.4 Interpolación temporal acotada (lineal / spline / circular)

Solo se imputan los **vacíos puntuales internos**. La función respeta el límite **por hueco completo**: un hueco de 5 h con límite de 3 h **no se rellena parcialmente** (a diferencia del parámetro `limit` de `pandas.interpolate`, que sí rellenaría las primeras 3 horas).

| Método | Cómo funciona | Variables |
|---|---|---|
| `linear` | recta entre el último y el primer dato válido que rodean el hueco | viento (m/s), humedad, oleaje, diagnóstico |
| `spline` | spline cúbico **local** (hasta 12 puntos válidos a cada lado); si no hay suficientes puntos, cae a lineal | temperatura, presión |
| `circular` | interpola seno y coseno por separado y recompone el ángulo | dirección del viento |

Tras interpolar, el resultado se **recorta al rango físico** de la variable (el spline puede sobrepasarlo). Cada hora imputada queda marcada en una **bandera** para que HU06/HU07 puedan distinguir dato medido de dato imputado.

In [ ]:
# ==== T09.4 - Imputación temporal acotada ====
from scipy.interpolate import CubicSpline

def _huecos_internos(serie):
    """Devuelve lista de (pos_ini, pos_fin) de los huecos de NaN que NO están en los bordes."""
    es_nulo = serie.isna().values
    n, i, out = len(es_nulo), 0, []
    while i < n:
        if es_nulo[i]:
            j = i
            while j + 1 < n and es_nulo[j + 1]:
                j += 1
            if i > 0 and j < n - 1:
                out.append((i, j))
            i = j + 1
        else:
            i += 1
    return out

def _lineal_en_hueco(valores, a, b):
    x0, x1 = a - 1, b + 1
    return np.interp(np.arange(a, b + 1), [x0, x1], [valores[x0], valores[x1]])

def _spline_en_hueco(valores, a, b, ventana):
    n = len(valores)
    valida = ~np.isnan(valores)
    izq = np.where(valida[max(0, a - ventana):a])[0] + max(0, a - ventana)
    der = np.where(valida[b + 1:min(n, b + 1 + ventana)])[0] + b + 1
    if len(izq) < 2 or len(der) < 2:
        return None
    xs = np.concatenate([izq, der])
    cs = CubicSpline(xs, valores[xs], bc_type='natural')
    return cs(np.arange(a, b + 1))

def imputar_acotado(serie, metodo, max_gap_h, rango, ventana=VENTANA_SPLINE_H):
    """Imputa SOLO huecos internos con longitud <= max_gap_h. Retorna (serie_imputada, mascara_imputado, metodo_efectivo_por_hueco)."""
    s = serie.copy()
    mask = pd.Series(False, index=s.index)
    usados = []
    if metodo == 'circular':
        rad = np.deg2rad(s)
        comp = {'sen': np.sin(rad), 'cos': np.cos(rad)}
        comp_imp = {}
        for k, c in comp.items():
            c_imp, m, _ = imputar_acotado(c, 'linear', max_gap_h, (-1, 1), ventana)
            comp_imp[k] = c_imp
            mask = mask | m
        ang = np.degrees(np.arctan2(comp_imp['sen'], comp_imp['cos'])) % 360
        s = s.where(~mask, ang)
        n_imp = int(((mask != mask.shift(fill_value=False)) & mask).sum())
        return s, mask, ['circular'] * n_imp
    valores = s.values.astype(float).copy()
    for a, b in _huecos_internos(serie):
        largo = b - a + 1
        if largo > max_gap_h:
            continue
        rel = None
        if metodo == 'spline':
            rel = _spline_en_hueco(valores, a, b, ventana)
            usados.append('spline' if rel is not None else 'lineal (respaldo)')
        if rel is None:
            rel = _lineal_en_hueco(valores, a, b)
            if metodo != 'spline': usados.append('lineal')
        valores[a:b + 1] = rel
        mask.iloc[a:b + 1] = True
    s = pd.Series(valores, index=serie.index, name=serie.name)
    lo, hi = rango
    s = s.where(~mask, s.clip(lo, hi))
    return s, mask, usados

df_limpio = df_rango.copy()
flags_imputacion = pd.DataFrame(index=df_limpio.index)
detalle_metodos = []

for v, cfg in CONFIG_VARIABLES.items():
    # La velocidad en mph NO se interpola aparte: se reconstruye desde m/s (evita inconsistencias entre columnas)
    s_imp, m_imp, usados = imputar_acotado(df_rango[v], cfg['metodo'], cfg['max_gap_h'], cfg['rango'])
    df_limpio[v] = s_imp
    flags_imputacion[f'{v}_imputado'] = m_imp
    detalle_metodos.append({'variable': v, 'metodo_configurado': cfg['metodo'], 'max_gap_h': cfg['max_gap_h'],
                            'huecos_imputados': len(usados), 'horas_imputadas': int(m_imp.sum()),
                            'metodo_efectivo': ', '.join(sorted(set(usados))) or '—'})

# Variable derivada en mph, coherente con m/s limpio
if 'velocidad_viento_mph' in df_h.columns:
    df_limpio['velocidad_viento_mph'] = df_limpio['velocidad_viento_ms'] / MPH_A_MS

resumen_imputacion = pd.DataFrame(detalle_metodos)
display(resumen_imputacion)

In [ ]:
# ==== Vista visual - Ejemplos de imputación (el hueco imputado más largo de cada método) ====
def graficar_ejemplo(ax, v, margen_h=18):
    m = flags_imputacion[f'{v}_imputado']
    if not m.any():
        ax.text(0.5, 0.5, f'{v}\n(sin huecos imputados)', ha='center', va='center', transform=ax.transAxes, color=COLOR_GRAY)
        ax.axis('off'); return
    grupo = (m != m.shift(fill_value=False)).cumsum()
    largos = m.groupby(grupo).sum()
    g_max = largos[largos == largos.max()].index[0]
    idx = m[(grupo == g_max) & m].index
    ini, fin = idx[0] - pd.Timedelta(hours=margen_h), idx[-1] + pd.Timedelta(hours=margen_h)
    ax.plot(df_rango.loc[ini:fin, v], color=COLOR_NAVY, lw=1.6, marker='o', ms=3, label='medido')
    ax.plot(df_limpio.loc[idx, v], color=COLOR_ORANGE, lw=0, marker='o', ms=6, label='imputado')
    ax.axvspan(idx[0], idx[-1], color=COLOR_GOLD, alpha=0.25, lw=0)
    ax.set_title(f'{v}  ·  {CONFIG_VARIABLES[v]["metodo"]}  ·  hueco de {len(idx)} h', fontsize=9.5,
                 color=COLOR_NAVY, fontweight='bold')
    ax.legend(fontsize=8, frameon=False); ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', labelsize=7.5, rotation=20)

ejemplos = ['velocidad_viento_ms', 'direccion_viento_deg', 'temperatura_c', 'altura_olas_m']
fig, axes = plt.subplots(1, 4, figsize=(18, 3.9))
for ax, v in zip(axes, ejemplos):
    graficar_ejemplo(ax, v)
fig.suptitle('T09.4 — Interpolación acotada: antes (azul) y después (naranja) en el hueco puntual más largo',
             fontsize=12.5, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

### T09.5 Vacíos prolongados: documentar y evaluar exclusión

Lo que **no** se pudo imputar (huecos prolongados y de borde) se trata con una regla explícita, en dos niveles:

1. **Exclusión de variable:** si tras imputar, el % de vacío de una variable supera su `max_pct_vacio`, se marca **EVALUAR EXCLUSIÓN DE VARIABLE**.
2. **Exclusión de bloque temporal:** una hora es **apta para modelado** solo si **todos los predictores** tienen dato (medido o imputado). Las horas con algún predictor vacío se agrupan en **bloques**, que se documentan con su causa probable:
   - **Falla de estación/datalogger:** casi todas las variables vacías a la vez, o diagnóstico también vacío.
   - **Falla de sensor individual:** solo algunas variables vacías.

Las filas **no se borran**: se marcan con `apto_modelado = False`. HU06 decidirá cómo construir rezagos sin cruzar estos bloques.

In [ ]:
# ==== T09.5 - Vacíos prolongados: bloques y exclusión ====
df_limpio['apto_modelado'] = df_limpio[PREDICTORES].notna().all(axis=1)
df_limpio['n_predictores_imputados'] = flags_imputacion[[f'{v}_imputado' for v in PREDICTORES]].sum(axis=1)

# --- Decisión por variable
decision_variables = []
for v, cfg in CONFIG_VARIABLES.items():
    pct_antes = 100 * df_crudo_h[v].isna().mean()
    pct_despues = 100 * df_limpio[v].isna().mean()
    exceso = pct_despues > cfg['max_pct_vacio']
    decision_variables.append({
        'variable': v, 'rol': cfg['rol'],
        'pct_vacio_antes': round(pct_antes, 2), 'pct_vacio_despues': round(pct_despues, 2),
        'completitud_despues_pct': round(100 - pct_despues, 2),
        'decision': 'EVALUAR EXCLUSIÓN DE VARIABLE' if exceso else 'CONSERVAR',
    })
decision_variables = pd.DataFrame(decision_variables)
display(decision_variables)

# --- Bloques temporales no aptos
no_apto = ~df_limpio['apto_modelado']
grupo = (no_apto != no_apto.shift(fill_value=False)).cumsum()
bloques = []
for _, b in df_limpio[no_apto].groupby(grupo[no_apto]):
    vacias = [v for v in PREDICTORES if df_limpio.loc[b.index, v].isna().all()]
    algun_vacio = [v for v in PREDICTORES if df_limpio.loc[b.index, v].isna().any()]
    diag_vacio = np.mean([df_limpio.loc[b.index, v].isna().mean() for v in DIAGNOSTICO]) if DIAGNOSTICO else 0
    if len(vacias) >= len(PREDICTORES) - 1 or diag_vacio >= 0.5:
        causa = 'Falla de estación / datalogger (vacío compartido)'
    else:
        causa = 'Falla de sensor individual'
    bloques.append({'inicio': b.index[0], 'fin': b.index[-1], 'horas': len(b),
                    'variables_afectadas': ', '.join(algun_vacio), 'causa_probable': causa,
                    'decision': 'Excluido del modelado (marcado apto_modelado=False)'})
bloques_excluidos = pd.DataFrame(bloques)
print(f'Bloques temporales no aptos: {len(bloques_excluidos)}  ·  horas no aptas: {int(no_apto.sum()):,} de {len(df_limpio):,} '
      f'({100 * no_apto.mean():.2f} %)')
if len(bloques_excluidos):
    display(bloques_excluidos.sort_values('horas', ascending=False).head(15).reset_index(drop=True))

### T09.6 Validación del tratamiento: completitud antes vs después

Criterio de aceptación de HU05: **completitud ≥ 90 %** o justificación del tratamiento. Se mide en dos niveles: **por variable** y **global** (horas con *todos* los predictores completos, que es lo realmente utilizable por el modelo).

In [ ]:
# ==== T09.6 - Completitud antes vs después ====
comp = decision_variables.set_index('variable')
completitud_global_antes = 100 * df_crudo_h[PREDICTORES].notna().all(axis=1).mean()
completitud_global_despues = 100 * df_limpio['apto_modelado'].mean()
print(f'Completitud GLOBAL (todos los predictores a la vez): antes = {completitud_global_antes:.2f} %  ->  después = {completitud_global_despues:.2f} %')
print(f'Criterio HU05: >= {UMBRAL_COMPLETITUD_PCT} %  ->  '
      f'{"CUMPLE" if completitud_global_despues >= UMBRAL_COMPLETITUD_PCT else "NO CUMPLE: justificar tratamiento (ver bloques prolongados)"}')

fig, ax = plt.subplots(figsize=(11, 4.6))
y = np.arange(len(comp))
ax.barh(y + 0.2, 100 - comp['pct_vacio_antes'], height=0.38, color=COLOR_GRAY, label='Antes')
ax.barh(y - 0.2, comp['completitud_despues_pct'], height=0.38, color=COLOR_TEAL, label='Después')
ax.axvline(UMBRAL_COMPLETITUD_PCT, color=COLOR_RED, ls='--', lw=1.4, label=f'Umbral {UMBRAL_COMPLETITUD_PCT} %')
ax.set_yticks(y); ax.set_yticklabels(comp.index, fontsize=9); ax.invert_yaxis()
ax.set_xlim(min(85, (100 - comp['pct_vacio_antes']).min() - 1), 100.3)
ax.set_xlabel('Completitud (%)'); ax.legend(frameon=False, fontsize=9, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.set_title('T09.6 — Completitud por variable: antes vs después de la imputación acotada',
             fontsize=12, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# ==== T09 - Control: la limpieza no distorsionó las distribuciones ====
filas = []
for v in VARIABLES:
    a, d = df_crudo_h[v].dropna(), df_limpio[v].dropna()
    filas.append({'variable': v, 'media_antes': a.mean(), 'media_despues': d.mean(),
                  'desv_antes': a.std(), 'desv_despues': d.std(),
                  'variacion_media_pct': 100 * (d.mean() - a.mean()) / a.mean() if a.mean() else np.nan})
control_dist = pd.DataFrame(filas).round(3)
display(control_dist)
print('Lectura: la media y la dispersión deben ser prácticamente iguales. Un cambio grande indicaría que se imputó demasiado.')

## 3.1.2 T10 — Corregir inconsistencias en horas de indisponibilidad

Las **horas de indisponibilidad** son la base de las dos variables objetivo (`horas_hasta_inicio_24h` y `duracion_indisponibilidad_h`). Un evento con fechas invertidas o una duración que no coincide con `fecha_fin − fecha_inicio` **contamina directamente las etiquetas**. Por eso T10 trabaja sobre el histórico de eventos, con reglas ordenadas y **una bitácora de cada cambio**.

### Reglas de T10 (en orden)

| # | Regla | Acción por defecto |
|---|---|---|
| R0 | Duplicados exactos y `id_evento` repetido | Conservar la **última** fila por `id_evento`; registrar si las filas repetidas **no** eran idénticas |
| R1 | `fecha_inicio` nula | Excluir (no hay evento definible) |
| R2 | `fecha_fin` nula | Según `POLITICA_FECHA_FIN_NULA`: por defecto `fin = inicio + duración` si `duracion_h_registrada` y `duracion_h_calculada` coinciden; si no, excluir |
| R3 | `fecha_fin < fecha_inicio` (fechas invertidas) | **Intercambiar** si eso hace coincidir con la duración registrada; si no, excluir y marcar para validar |
| R4 | `duracion_h_registrada` ≠ `fecha_fin − fecha_inicio` | La **fuente de verdad son las fechas** (de ahí salen los targets); se corrige la duración y se registra |
| R5 | Duración ≤ 0 | Excluir |
| R6 | Duración atípica (Q3 + 3·IQR) | **Solo marcar**; un evento largo puede ser real |
| R7 | Formato de `variable_dominante` | Normalizar (minúsculas, sin tildes, variantes unificadas) |
| R8 | Eventos solapados | Marcar (o fusionar si `FUSIONAR_SOLAPADOS=True`) |
| R9 | Causa no meteo-oceanográfica | Excluir según `causa_registrada` y `variable_dominante` (se cruzan ambas); si no hubiera columna de causa, queda pendiente de validar con Operaciones (D02/Q04) |

> **Aviso anti data-leakage (Q07):** las columnas `*_durante_evento_*` del histórico (oleaje/viento máximo, temperatura, humedad, presión durante el evento) son lecturas **posteriores a `t`**. Sirven para caracterizar y validar el evento (Rol B), **nunca como predictores**. HU06 no debe incluirlas en `X`.
>
> **Principio:** cada decisión que **no** es una corrección mecánica obvia queda marcada con `requiere_validacion = True`, para llevarla al Product Owner en lugar de decidirla en silencio.

In [ ]:
# ==== T10 - Preparación de la bitácora y utilidades ====
bitacora_eventos = []

def registrar(id_evento, regla, campo, antes, despues, accion, requiere_validacion=False):
    bitacora_eventos.append({'id_evento': id_evento, 'regla': regla, 'campo': campo, 'valor_antes': antes,
                             'valor_despues': despues, 'accion': accion, 'requiere_validacion': requiere_validacion})

def _norm_texto(x):
    if pd.isna(x): return x
    t = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', t).strip().lower()

ev = df_eventos_crudo.copy().reset_index(drop=True)
ev['_orden_original'] = np.arange(len(ev))
embudo = [('Eventos en el histórico crudo', len(ev))]

In [ ]:
# ==== R0 / R1 / R2 - Duplicados y nulos en eventos (parte T09 aplicada al histórico) ====
# R0a: filas completamente idénticas
n_ident = int(ev.drop(columns='_orden_original').duplicated().sum())
if n_ident:
    for i in ev[ev.drop(columns='_orden_original').duplicated(keep='last')]['id_evento']:
        registrar(i, 'R0', 'fila', 'duplicada exacta', 'eliminada', 'Se conserva una sola copia')
ev = ev.drop_duplicates(subset=[c for c in ev.columns if c != '_orden_original'], keep='last')

# R0b: id_evento repetido con contenido distinto -> se conserva la ÚLTIMA y se avisa del conflicto
dup_id = ev[ev.duplicated('id_evento', keep=False)].sort_values(['id_evento', '_orden_original'])
for i, g in dup_id.groupby('id_evento'):
    registrar(i, 'R0', 'id_evento', f'{len(g)} filas con distinto contenido', 'se conserva la última',
              'Dato digitado más de una vez con valores distintos', requiere_validacion=True)
n_dup_id = int(ev.duplicated('id_evento').sum())
ev = ev.sort_values('_orden_original').drop_duplicates('id_evento', keep='last')
embudo.append(('Tras eliminar duplicados (R0)', len(ev)))

# R1: fecha_inicio nula
m = ev['fecha_inicio'].isna()
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R1', 'fecha_inicio', 'nula', 'excluido', 'Sin inicio no se puede definir el evento')
n_ini_nulo = int(m.sum()); ev = ev[~m]

# R2: fecha_fin nula
m = ev['fecha_fin'].isna()
n_fin_nulo = int(m.sum())
if 'duracion_h_registrada' in ev.columns and POLITICA_FECHA_FIN_NULA == 'imputar_con_duracion_registrada':
    ok = m & ev['duracion_h_registrada'].gt(0)
    if 'duracion_h_calculada' in ev.columns:      # solo si las dos duraciones del CSV coinciden entre sí
        ok &= (ev['duracion_h_calculada'] - ev['duracion_h_registrada']).abs() <= TOL_DURACION_H
    for i, ini, d in ev.loc[ok, ['id_evento', 'fecha_inicio', 'duracion_h_registrada']].itertuples(index=False):
        registrar(i, 'R2', 'fecha_fin', 'nula', str(ini + pd.Timedelta(hours=d)), 'fin = inicio + duración (registrada = calculada)', True)
    ev.loc[ok, 'fecha_fin'] = ev.loc[ok, 'fecha_inicio'] + pd.to_timedelta(ev.loc[ok, 'duracion_h_registrada'], unit='h')
    m = ev['fecha_fin'].isna()
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R2', 'fecha_fin', 'nula', 'excluido', 'Evento en curso al exportar: no se conoce la duración real', True)
ev = ev[~m]
embudo.append(('Tras tratar fechas nulas (R1-R2)', len(ev)))

print(f'R0  duplicados exactos: {n_ident}  ·  id_evento repetidos: {n_dup_id}')
print(f'R1  fecha_inicio nula: {n_ini_nulo}  ·  R2 fecha_fin nula: {n_fin_nulo} (política: {POLITICA_FECHA_FIN_NULA})')

In [ ]:
# ==== R3 / R4 / R5 / R6 - Fechas invertidas y duración ====
ev = ev.copy()
tiene_dur_reg = 'duracion_h_registrada' in ev.columns
dur_fechas = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)

# R3: fechas invertidas
inv = dur_fechas < 0
n_swap = n_inv_excl = 0
for idx in ev.index[inv]:
    ini, fin = ev.at[idx, 'fecha_inicio'], ev.at[idx, 'fecha_fin']
    dur_swap = (ini - fin) / pd.Timedelta(hours=1)
    reg = ev.at[idx, 'duracion_h_registrada'] if tiene_dur_reg else np.nan
    if tiene_dur_reg and pd.notna(reg) and abs(dur_swap - reg) <= TOL_DURACION_H:
        ev.at[idx, 'fecha_inicio'], ev.at[idx, 'fecha_fin'] = fin, ini
        registrar(ev.at[idx, 'id_evento'], 'R3', 'fecha_inicio/fecha_fin', f'{ini} / {fin}', f'{fin} / {ini}',
                  'Fechas intercambiadas: coincide con la duración registrada')
        n_swap += 1
    else:
        registrar(ev.at[idx, 'id_evento'], 'R3', 'fecha_inicio/fecha_fin', f'{ini} / {fin}', 'excluido',
                  'Fechas invertidas y no coincide con la duración registrada: validar con Operaciones', True)
        ev.at[idx, '_excluir'] = True
if '_excluir' in ev.columns:
    n_inv_excl = int(ev['_excluir'].fillna(False).sum())
    ev = ev[~ev['_excluir'].fillna(False).astype(bool)].drop(columns='_excluir')
embudo.append(('Tras corregir fechas invertidas (R3)', len(ev)))

# R4: duración registrada vs fin - inicio (la fuente de verdad son las fechas)
ev['duracion_h'] = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)
n_dur_corr = n_dur_revisar = 0
if tiene_dur_reg:
    dif = (ev['duracion_h'] - ev['duracion_h_registrada']).abs()
    mal = dif > TOL_DURACION_H
    for idx in ev.index[mal | ev['duracion_h_registrada'].isna()]:
        reg, calc, d = ev.at[idx, 'duracion_h_registrada'], ev.at[idx, 'duracion_h'], dif.at[idx]
        revisar = bool(pd.notna(d) and d > UMBRAL_REVISION_H)
        registrar(ev.at[idx, 'id_evento'], 'R4', 'duracion_h', reg, round(calc, 3),
                  'Duración recalculada como fecha_fin - fecha_inicio', revisar)
        n_dur_corr += 1; n_dur_revisar += int(revisar)

# R5: duración <= 0
m = ev['duracion_h'] <= 0
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R5', 'duracion_h', '<= 0', 'excluido', 'Duración nula: no es un evento utilizable')
ev = ev[~m]
embudo.append(('Tras eliminar duraciones <= 0 (R5)', len(ev)))

# R6: duración atípica (solo marca)
q1, q3 = ev['duracion_h'].quantile([.25, .75]); lim_atip = q3 + 3 * (q3 - q1)
ev['duracion_atipica'] = ev['duracion_h'] > lim_atip
for i, d in ev.loc[ev['duracion_atipica'], ['id_evento', 'duracion_h']].itertuples(index=False):
    registrar(i, 'R6', 'duracion_h', round(d, 2), round(d, 2), f'Duración atípica (> {lim_atip:.1f} h): se conserva y se marca', True)

print(f'R3  fechas invertidas: intercambiadas = {n_swap}, excluidas = {n_inv_excl}')
if 'duracion_h_calculada' in ev.columns:
    print(f"R4  info: duracion_h_calculada (del CSV) difiere de fin-inicio en {int(((ev['duracion_h'] - ev['duracion_h_calculada']).abs() > TOL_DURACION_H).sum())} eventos; se recalcula siempre desde las fechas")
print(f'R4  duraciones corregidas: {n_dur_corr}  (de ellas {n_dur_revisar} con diferencia > {UMBRAL_REVISION_H} h -> validar)')
print(f'R6  duraciones atípicas marcadas (> {lim_atip:.1f} h): {int(ev["duracion_atipica"].sum())}')

In [ ]:
# ==== R7 / R8 / R9 - Formato, solapes y causa ====
# R7: normalizar variable_dominante
if 'variable_dominante' in ev.columns:
    antes = ev['variable_dominante'].nunique()
    base = ev['variable_dominante'].apply(_norm_texto)
    def _unificar(t):
        if pd.isna(t): return t
        if re.search(r'^no |no meteo', t):        return 'no_meteo_oceanografica'
        if re.search(r'ola|oleaj|swell|wave', t): return 'oleaje'
        if re.search(r'viento|wind', t):          return 'viento'
        return t
    ev['variable_dominante'] = base.apply(_unificar)
    despues = ev['variable_dominante'].nunique()
    print(f'R7  variantes de variable_dominante: {antes} -> {despues}')
    display(ev['variable_dominante'].value_counts(dropna=False).rename('eventos').to_frame())
    print('Revisar manualmente cualquier categoría distinta de viento/oleaje/no_meteo_oceanografica: no se unifica sin criterio de Operaciones.')

# R8: solapes
ev = ev.sort_values('fecha_inicio').reset_index(drop=True)
fin_max_previo = ev['fecha_fin'].cummax().shift()
ev['solapa_con_anterior'] = ev['fecha_inicio'] < fin_max_previo
n_solapes = int(ev['solapa_con_anterior'].sum())
if n_solapes and FUSIONAR_SOLAPADOS:
    fusion, actual = [], None
    for r in ev.to_dict('records'):
        if actual is not None and r['fecha_inicio'] < actual['fecha_fin']:
            registrar(r['id_evento'], 'R8', 'evento', 'solapado', f"fusionado con {actual['id_evento']}", 'Eventos fusionados', True)
            actual['fecha_fin'] = max(actual['fecha_fin'], r['fecha_fin'])
        else:
            if actual is not None: fusion.append(actual)
            actual = r
    fusion.append(actual)
    ev = pd.DataFrame(fusion)
    ev['duracion_h'] = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)
    ev['solapa_con_anterior'] = False
elif n_solapes:
    for i in ev.loc[ev['solapa_con_anterior'], 'id_evento']:
        registrar(i, 'R8', 'evento', 'solapa con el anterior', 'marcado', 'Solo marcado (FUSIONAR_SOLAPADOS=False)', True)
print(f'R8  eventos solapados: {n_solapes}  (fusionar = {FUSIONAR_SOLAPADOS})')

# R9: causa no meteo-oceanográfica
col_causa = next((c for c in COLUMNAS_CAUSA_CANDIDATAS if c in ev.columns), None)
n_no_clim = 0
if col_causa:
    texto = ev[col_causa].apply(_norm_texto).fillna('')
    no_clim_causa = texto.apply(lambda t: any(p in t for p in PALABRAS_CAUSA_NO_CLIMATICA))
    no_clim_dom = ev['variable_dominante'].eq('no_meteo_oceanografica') if 'variable_dominante' in ev.columns else pd.Series(False, index=ev.index)
    discrepancias = int((no_clim_causa != no_clim_dom).sum())
    no_clim = no_clim_causa | no_clim_dom
    print(f'    Cruce causa_registrada vs variable_dominante: {discrepancias} discrepancias (0 = ambas fuentes coinciden)')
    display(ev.assign(_causa=texto)[no_clim].groupby(col_causa).size().rename('eventos_excluidos').to_frame())
    for i, c in ev.loc[no_clim, ['id_evento', col_causa]].itertuples(index=False):
        registrar(i, 'R9', col_causa, c, 'excluido', 'Causa no meteo-oceanográfica', False)
    n_no_clim = int(no_clim.sum()); ev = ev[~no_clim]
    display(ev.get('variable_dominante', pd.Series(dtype=str)).value_counts().rename('eventos_conservados').to_frame())
    print(f'R9  columna de causa usada: {col_causa!r}  ->  eventos excluidos por causa no climática: {n_no_clim}')
else:
    print('R9  El histórico NO trae una columna de causa reconocible: la depuración de eventos no meteo-oceanográficos '
          'queda PENDIENTE de validar con Operaciones (control Q04 / decisión D02).')
embudo.append(('Eventos limpios finales', len(ev)))

# Cobertura de sensores (informativo: no se excluye)
cob_ini, cob_fin = df_limpio.index.min(), df_limpio.index.max()
ev['en_cobertura_sensores'] = (ev['fecha_fin'] >= cob_ini) & (ev['fecha_inicio'] <= cob_fin)
print(f'Eventos dentro del periodo con sensores ({cob_ini:%Y-%m-%d} a {cob_fin:%Y-%m-%d}): {int(ev["en_cobertura_sensores"].sum())} de {len(ev)}')

df_eventos_limpio = ev.drop(columns=['_orden_original'], errors='ignore').reset_index(drop=True)
bitacora_correcciones_eventos = pd.DataFrame(bitacora_eventos)

In [ ]:
# ==== Vista visual - Embudo de depuración de eventos + resumen de reglas ====
etq, val = zip(*embudo)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
y = np.arange(len(etq))
ax.barh(y, val, color=[COLOR_GRAY] + [COLOR_TEAL] * (len(val) - 2) + [COLOR_NAVY])
for yi, v in zip(y, val): ax.text(v, yi, f' {v}', va='center', fontsize=9, color=COLOR_NAVY, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(etq, fontsize=8.8); ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False); ax.set_xlabel('N.° de eventos')
ax.set_title('Embudo de depuración del histórico', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')

ax = axes[1]
if len(bitacora_correcciones_eventos):
    r = bitacora_correcciones_eventos.groupby('regla').size()
    ax.bar(r.index, r.values, color=COLOR_ORANGE)
    for xi, v in zip(r.index, r.values): ax.text(xi, v, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLOR_NAVY)
else:
    ax.text(0.5, 0.5, 'Sin correcciones registradas', ha='center', va='center', transform=ax.transAxes, color=COLOR_GRAY)
ax.spines[['top', 'right']].set_visible(False); ax.set_ylabel('N.° de registros en bitácora')
ax.set_title('Intervenciones por regla (T10)', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')
fig.suptitle('T10 — Corrección de inconsistencias en horas de indisponibilidad', fontsize=13, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

if len(bitacora_correcciones_eventos):
    pend = bitacora_correcciones_eventos[bitacora_correcciones_eventos['requiere_validacion']]
    print(f'Registros que requieren validación con Operaciones / Product Owner: {len(pend)}')
    display(pend.head(15))

### T10.1 Reconstrucción de los targets con los eventos limpios

Se vuelven a construir `ocurre_evento_24h`, `horas_hasta_inicio_24h` y `duracion_indisponibilidad_h` sobre la malla limpia. La función implementa **exactamente la definición de HU01/T02**: para cada origen `t` se toma el evento **más próximo** con inicio en `(t, t+24h]`; si no existe, los targets quedan en **NaN** (no se fuerza a 0, regla Q10).

> Si en tu sesión ya existe `construir_targets_evento()` del Sprint 1, puedes usarla: devuelve las mismas columnas. Aquí se incluye una versión **autocontenida y vectorizada** para que esta parte corra por sí sola.

In [ ]:
# ==== T10.1 - Targets con eventos limpios ====
def construir_targets_hu05(timestamps, df_eventos, horizonte_h=HORIZONTE_H):
    ev_ord = df_eventos.sort_values('fecha_inicio')
    inicios = ev_ord['fecha_inicio'].values.astype('datetime64[ns]')
    duraciones = ((ev_ord['fecha_fin'] - ev_ord['fecha_inicio']) / pd.Timedelta(hours=1)).values
    t = pd.DatetimeIndex(timestamps).values.astype('datetime64[ns]')
    pos = np.searchsorted(inicios, t, side='right')                 # primer inicio estrictamente > t
    hay = pos < len(inicios)
    pos_c = np.where(hay, pos, 0)
    h_hasta = (inicios[pos_c] - t) / np.timedelta64(1, 'h') if len(inicios) else np.full(len(t), np.nan)
    en_ventana = hay & (h_hasta <= horizonte_h)
    return pd.DataFrame({
        'ocurre_evento_24h': en_ventana.astype(int),
        'horas_hasta_inicio_24h': np.where(en_ventana, h_hasta, np.nan),
        'duracion_indisponibilidad_h': np.where(en_ventana, duraciones[pos_c], np.nan),
    }, index=pd.DatetimeIndex(timestamps))

# Comparación: filtro provisional de HU03 vs depuración completa de HU05
prov = (df_eventos_crudo.dropna(subset=['fecha_inicio', 'fecha_fin']).drop_duplicates('id_evento'))
prov = prov[prov['fecha_fin'] > prov['fecha_inicio']]
t_prov = construir_targets_hu05(df_limpio.index, prov)
t_limpio = construir_targets_hu05(df_limpio.index, df_eventos_limpio)

comparacion_targets = pd.DataFrame({
    'Filtro provisional (HU03)': [len(prov), int(t_prov['ocurre_evento_24h'].sum()), round(t_prov['duracion_indisponibilidad_h'].mean(), 2)],
    'Depuración completa (HU05)': [len(df_eventos_limpio), int(t_limpio['ocurre_evento_24h'].sum()), round(t_limpio['duracion_indisponibilidad_h'].mean(), 2)],
}, index=['Eventos utilizados', 'Horas con evento en las próximas 24 h', 'Duración media del evento (h)'])
display(comparacion_targets)

df_limpio = df_limpio.join(t_limpio)

## Verificaciones finales, exportación y Definition of Done

Se ejecutan **comprobaciones automáticas** sobre el dataset resultante y se exportan los entregables de HU05.

In [ ]:
# ==== Verificaciones finales (assertions) ====
chequeos = {
    'Índice horario único y ordenado': df_limpio.index.is_unique and df_limpio.index.is_monotonic_increasing,
    'Malla horaria continua (sin saltos)': (df_limpio.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all(),
    'Sin valores fuera de rango físico en ninguna variable': all(
        df_limpio[v].dropna().between(*CONFIG_VARIABLES[v]['rango']).all() for v in VARIABLES),
    'Imputación solo en horas originalmente vacías': all(
        df_rango.loc[flags_imputacion[f'{v}_imputado'], v].isna().all() for v in VARIABLES),
    'Eventos: id_evento único': df_eventos_limpio['id_evento'].is_unique,
    'Eventos: fecha_fin > fecha_inicio en todos': (df_eventos_limpio['fecha_fin'] > df_eventos_limpio['fecha_inicio']).all(),
    'Eventos: duracion_h = fin - inicio': ((df_eventos_limpio['fecha_fin'] - df_eventos_limpio['fecha_inicio']) / pd.Timedelta(hours=1) - df_eventos_limpio['duracion_h']).abs().max() < 1e-6,
    'Targets: horas_hasta_inicio_24h dentro de (0, 24]': t_limpio['horas_hasta_inicio_24h'].dropna().between(0, HORIZONTE_H, inclusive='right').all(),
    'Targets: sin 0 forzado donde no hay evento (NaN)': t_limpio.loc[t_limpio['ocurre_evento_24h'] == 0, 'horas_hasta_inicio_24h'].isna().all(),
}
# Verificación estricta de huecos imputados: ninguno más largo que el límite
for v in VARIABLES:
    m = flags_imputacion[f'{v}_imputado']
    g = (m != m.shift(fill_value=False)).cumsum()
    mx = int(m.groupby(g).sum().max()) if m.any() else 0
    chequeos[f'Máx. racha imputada {v} <= {CONFIG_VARIABLES[v]["max_gap_h"]} h'] = mx <= CONFIG_VARIABLES[v]['max_gap_h']

tabla_chequeos = pd.DataFrame({'Verificación': list(chequeos), 'Resultado': ['OK' if bool(v) else 'FALLA' for v in chequeos.values()]})
display(tabla_chequeos)
assert (tabla_chequeos['Resultado'] == 'OK').all(), 'Hay verificaciones que fallan: revisar la tabla anterior.'

In [ ]:
# ==== Exportación de entregables de HU05 ====
columnas_modelo = (['velocidad_viento_mph'] if 'velocidad_viento_mph' in df_limpio.columns else []) + VARIABLES
columnas_modelo = list(dict.fromkeys(columnas_modelo))
dataset_limpio = df_limpio[columnas_modelo + ['apto_modelado', 'n_predictores_imputados',
                                              'ocurre_evento_24h', 'horas_hasta_inicio_24h', 'duracion_indisponibilidad_h']]

dataset_limpio.to_csv('dataset_limpio_horario_HU05.csv')
flags_imputacion.to_csv('flags_imputacion_HU05.csv')
df_eventos_limpio.to_csv('eventos_limpios_HU05.csv', index=False)
bitacora_vacios.to_csv('bitacora_vacios_HU05.csv', index=False)
bitacora_correcciones_eventos.to_csv('bitacora_correcciones_eventos_HU05.csv', index=False)
decision_variables.to_csv('decision_variables_HU05.csv', index=False)

print('Archivos generados:')
for f in ['dataset_limpio_horario_HU05.csv', 'flags_imputacion_HU05.csv', 'eventos_limpios_HU05.csv',
          'bitacora_vacios_HU05.csv', 'bitacora_correcciones_eventos_HU05.csv', 'decision_variables_HU05.csv']:
    print(f'  - {f}  ({Path(f).stat().st_size/1024:,.0f} KB)')

try:
    from google.colab import files
    # files.download('dataset_limpio_horario_HU05.csv')   # descomentar para descargar
except ImportError:
    pass

### Definition of Done — HU05 (T09 / T10)

La tabla se **calcula con los resultados reales** de esta ejecución (no está escrita a mano).

In [ ]:
# ==== HU05 - Definition of Done (calculada) ====
def estado(c): return 'CUMPLE' if c else 'NO CUMPLE'
n_vars_exclusion = int((decision_variables['decision'] != 'CONSERVAR').sum())
dod_hu05 = pd.DataFrame([
    ['T09', 'Verificación y deduplicación por marca de tiempo documentada (CSV crudos e índice horario)', 'CUMPLE'],
    ['T09', 'Vacíos puntuales imputados con interpolación lineal/spline/circular acotada por variable', estado((tabla_chequeos['Resultado'] == 'OK').all())],
    ['T09', 'Vacíos prolongados documentados en bitácora, sin imputar', 'CUMPLE'],
    ['T09', f'Completitud global >= {UMBRAL_COMPLETITUD_PCT} % (actual: {completitud_global_despues:.2f} %)', estado(completitud_global_despues >= UMBRAL_COMPLETITUD_PCT)],
    ['T09', f'Variables a evaluar para exclusión: {n_vars_exclusion}', 'CUMPLE' if n_vars_exclusion == 0 else 'REVISAR'],
    ['T09', 'Banderas de imputación exportadas (dato medido vs imputado)', 'CUMPLE'],
    ['T10', 'Fechas invertidas y duraciones inconsistentes corregidas con bitácora', 'CUMPLE'],
    ['T10', 'Targets reconstruidos con eventos limpios', 'CUMPLE'],
    ['T10', 'Depuración de eventos por causa no meteo-oceanográfica',
     'CUMPLE' if col_causa else 'PENDIENTE (validar con Operaciones: no hay columna de causa)'],
    ['-', 'Decisiones que requieren validación del Product Owner listadas en la bitácora',
     f'{int(bitacora_correcciones_eventos["requiere_validacion"].sum()) if len(bitacora_correcciones_eventos) else 0} registros'],
], columns=['Tarea', 'Criterio de aceptación', 'Estado'])
display(dod_hu05)

### Decisiones abiertas para validar (llevar a la asesora / Product Owner)

1. **Límites de interpolación por variable** (`max_gap_h`): son valores de partida; confirmar o ajustar con el diagnóstico de T09.3 y el criterio de Operaciones.
2. **`fecha_fin` nula:** se imputa como inicio + duración cuando las dos duraciones del CSV coinciden; confirmar con Operaciones (la alternativa es excluir el evento).
3. **Fuente de verdad de la duración:** se asumen las **fechas** (de ahí salen los targets); confirmar que la duración digitada no es la correcta en los casos con diferencia > 1 h.
4. **Eventos de causa no meteo-oceanográfica:** requieren una columna de causa en el histórico, o la validación directa con Operaciones (D02/Q04).
5. **Solapes entre eventos:** por defecto solo se marcan; decidir si se fusionan.

### Puente hacia HU06 (Ingeniería de características)

HU06 debe **respetar `apto_modelado`** y las banderas de imputación al construir los rezagos (t-1, t-2, t-3, t-6, t-12, t-24) y las ventanas móviles de 6 y 24 h: una ventana que cruza un bloque excluido no debe tratarse como dato continuo.

## 3.2 HU06 — Ingeniería de características

**Historia de usuario (TB2):** *Como Científico de datos, quiero transformar las variables meteorológicas y oceanográficas, para mejorar el aprendizaje del modelo.*

| N.° | Tarea | Contenido |
|---|---|---|
| 3.2.1 | **T11** (parte 1) | Seleccionar, descartar y **codificar** variables (dirección del viento → seno y coseno) |
| 3.2.2 | **T12** | Generar **variables derivadas** meteo-oceanográficas: rezagos, medias/desviaciones móviles, tasas de cambio y proporción imputada |
| 3.2.3 | **T11** (parte 2) | **Partición temporal** (entrenamiento / validación / prueba) y **normalización ajustada solo con entrenamiento** |

> **¿Por qué T11 aparece partido en dos?** La *codificación* no aprende nada de los datos (es una fórmula), así que va antes. La *normalización* sí aprende (media y desviación) y debe cubrir también las variables derivadas de T12, por eso se ajusta al final y **solo con el conjunto de entrenamiento**. Ajustarla antes de partir, o con todo el histórico, filtraría información del futuro hacia el entrenamiento (*data leakage*).

### Alineación con el TB2

| Elemento | Lo que dice el TB2 | Cómo se implementa aquí |
|---|---|---|
| Insumo | Dataset preparado en HU05 (Sprint 2) | `dataset_limpio_horario_HU05.csv` |
| Rezagos (inciso g) | t-1, t-2, t-3, t-6 (corto plazo) y t-12, t-24 (mediano plazo) | `LAGS_H = [1, 2, 3, 6, 12, 24]` |
| Estadísticos móviles (inciso g) | Media y desviación estándar en ventanas de 6 y 24 h | `VENTANAS_H = [6, 24]` |
| Validación temporal (inciso h) | *Walk-forward*: el entrenamiento solo usa información anterior al corte | Partición cronológica con purga de 24 h + 4 *folds* walk-forward |
| Dos modelos | **Clasificador** de ocurrencia (F1-score y AUC) + **regresor de doble salida** (`horas_hasta_inicio_24h`, `duracion_indisponibilidad_h`; MAE, RMSE, MAPE, R²) | Dos tablas de entrenamiento derivadas del mismo dataset: `obtener_datos(..., 'clasificacion')` y `obtener_datos(..., 'regresion')` |
| Criterio de HU06 (DoD) | Documentar normalización/codificación, variables derivadas y partición train/test; dataset reproducible y apto para entrenamiento | Catálogo de variables, JSON de configuración, escalador serializado y verificaciones automáticas |

### Decisiones de HU05 ya validadas (HU06 las hereda; no se modificó ninguna celda de HU05)

| Decisión abierta de HU05 | Respuesta | Efecto en HU06 |
|---|---|---|
| Límites de interpolación por variable (`max_gap_h`) | La tolerancia actual es adecuada | Se mantiene el dataset HU05 tal cual; la proporción de datos imputados por ventana se mide en T12 |
| `fecha_fin` nula en el histórico | El método automático es aceptable | Los *targets* de HU05 ya incorporan ese tratamiento |
| Fuente de verdad de la duración | Se confía en las fechas y horas registradas por los sistemas del terminal | `duracion_indisponibilidad_h` = `fecha_fin − fecha_inicio` (sin cambios) |
| Eventos de causa no meteo-oceanográfica | Se excluyen | Ya excluidos en HU05 (R9); HU06 no recibe esos eventos |
| Solapes entre eventos | Mantener `FUSIONAR_SOLAPADOS = False` (solo marcar) | Sin cambios en los *targets* |

### Vista rápida: el flujo completo de HU06

In [ ]:
# ==== HU06 - Preparación del entorno y configuración (EDITABLE) ====
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch
from IPython.display import display
import joblib
import sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 60)

# --- Estilo: coincide con HU05 (si HU05 ya corrió en esta sesión, se reutilizan sus colores)
COLOR_NAVY    = globals().get('COLOR_NAVY',    '#1B3A5C')
COLOR_TEAL    = globals().get('COLOR_TEAL',    '#2A9D8F')
COLOR_TEAL_LT = globals().get('COLOR_TEAL_LT', '#D6EFEC')
COLOR_ORANGE  = globals().get('COLOR_ORANGE',  '#E9762B')
COLOR_RED     = globals().get('COLOR_RED',     '#C43D3D')
COLOR_GRAY    = globals().get('COLOR_GRAY',    '#9AA5B1')
COLOR_GRAY_LT = globals().get('COLOR_GRAY_LT', '#EEF1F4')
COLOR_GOLD    = globals().get('COLOR_GOLD',    '#E3A857')
plt.rcParams['font.family'] = 'DejaVu Sans'

def caja(ax, x, y, w, h, texto, color=COLOR_NAVY, texto_color='white', fontsize=10, fontweight='bold'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.05',
                                linewidth=1.5, edgecolor=color, facecolor=color, zorder=2))
    ax.text(x + w/2, y + h/2, texto, ha='center', va='center', fontsize=fontsize,
            color=texto_color, fontweight=fontweight, zorder=3, linespacing=1.35)

def flecha(ax, p1, p2, color=COLOR_NAVY, lw=1.8, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=15, color=color,
                                 lw=lw, connectionstyle=f'arc3,rad={rad}', zorder=1))

def lienzo(w=11, h=5, xlim=(0, 11), ylim=(0, 5)):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis('off')
    return fig, ax

# ---------------------------------------------------------------- parámetros
RUTA_DATASET = 'dataset_limpio_horario_HU05.csv'      # salida de HU05 (insumo de HU06)
HORIZONTE_H  = globals().get('HORIZONTE_H', 24)       # horizonte de predicción (HU01/T02)
SEMILLA      = 42

# ---- T11: selección, codificación y normalización
VARS_CONTINUAS = ['velocidad_viento_ms', 'temperatura_c', 'humedad_relativa_pct',
                  'presion_atmosferica_hpa', 'altura_olas_m']
VAR_DIRECCION  = 'direccion_viento_deg'               # variable circular -> seno y coseno
COLUMNAS_DESCARTADAS = {
    'velocidad_viento_mph':             'Duplicada: es la misma variable que velocidad_viento_ms (se conserva m/s, unidad de la literatura)',
    'temperatura_interna_datalogger_c': 'Diagnóstico del equipo (HU05): no es predictor',
    'voltaje_datalogger_v':             'Diagnóstico del equipo (HU05): no es predictor',
}
ESCALADOR = 'standard'                                # 'standard' (z-score) | 'minmax' | 'robust'

# ---- T12: variables derivadas (TB2, inciso g)
LAGS_H         = [1, 2, 3, 6, 12, 24]                 # rezagos
VENTANAS_H     = [6, 24]                              # media y desviación móvil
TASAS_CAMBIO_H = [3, 6]                               # x(t) - x(t-k): tendencia (p. ej. caída de presión)
MAX_FRAC_IMPUTADA_VENTANA = 0.5                       # ventana (24 h) con más de este % imputado -> no entrenable
INCLUIR_FRAC_IMPUTADA_EN_X = False                    # True: la proporción imputada también entra como predictor

# ---- Partición temporal (TB2, inciso h)
FRAC_TRAIN, FRAC_VAL = 0.70, 0.15                     # el resto (15 %) es prueba final
GAP_H       = HORIZONTE_H                             # purga: la etiqueta de t mira hasta t+24 h
N_FOLDS_WF  = 4                                       # ventanas walk-forward para medir estabilidad (HU08)

TARGETS = ['ocurre_evento_24h', 'horas_hasta_inicio_24h', 'duracion_indisponibilidad_h']
TARGET_CLF, TARGETS_REG = TARGETS[0], TARGETS[1:]
N_PRED = len(VARS_CONTINUAS) + 1                      # predictores meteo-oceanográficos de HU05 (incluye dirección)
MIN_RACHA_H = max(max(LAGS_H) + 1, max(VENTANAS_H), max(TASAS_CAMBIO_H) + 1)   # horas aptas consecutivas necesarias

display(pd.DataFrame([
    ['T11', 'Variables continuas', ', '.join(VARS_CONTINUAS)],
    ['T11', 'Variable circular (seno/coseno)', VAR_DIRECCION],
    ['T11', 'Columnas descartadas', ', '.join(COLUMNAS_DESCARTADAS)],
    ['T11', 'Escalador (ajustado solo con entrenamiento)', ESCALADOR],
    ['T12', 'Rezagos (h)', str(LAGS_H)],
    ['T12', 'Ventanas móviles (h): media y desviación', str(VENTANAS_H)],
    ['T12', 'Tasas de cambio (h)', str(TASAS_CAMBIO_H)],
    ['T12', 'Horas aptas consecutivas exigidas', f'{MIN_RACHA_H} h (el rezago más largo necesita t-24 … t)'],
    ['T12', 'Máx. proporción imputada en la ventana de 24 h', f'{MAX_FRAC_IMPUTADA_VENTANA:.0%}'],
    ['T11', 'Partición temporal', f'{FRAC_TRAIN:.0%} entrenamiento / {FRAC_VAL:.0%} validación / {1-FRAC_TRAIN-FRAC_VAL:.0%} prueba, con purga de {GAP_H} h'],
    ['T11', 'Folds walk-forward', str(N_FOLDS_WF)],
], columns=['Tarea', 'Parámetro', 'Valor']))
print('Entorno HU06 listo.')

In [ ]:
# ==== Vista rápida del flujo HU06 ====
fig, ax = lienzo(13.2, 6.0, xlim=(0, 13.2), ylim=(0, 6.0))
ax.set_title('HU06 — Ingeniería de características: del dataset limpio al dataset entrenable',
             fontsize=13, color=COLOR_NAVY, fontweight='bold', pad=12)

caja(ax, 0.2, 4.35, 2.4, 1.15, 'Entrada (HU05)\ndataset_limpio_\nhorario_HU05.csv', color=COLOR_GRAY, fontsize=8.8)
flecha(ax, (2.6, 4.92), (3.1, 4.92))
ax.text(3.1, 5.68, 'T11 · Codificar y depurar variables', fontsize=9.5, color=COLOR_TEAL, fontweight='bold')
for i, t in enumerate(['T11.1 Descartar\nmph y datalogger\n(duplicada / diagnóstico)',
                       'T11.2 Dirección del\nviento → seno y\ncoseno (circular)']):
    x = 3.1 + i * 2.75
    caja(ax, x, 4.35, 2.45, 1.15, t, color=COLOR_TEAL, fontsize=8.4)
    if i == 0: flecha(ax, (x + 2.45, 4.92), (x + 2.75, 4.92), color=COLOR_TEAL)

ax.text(0.2, 3.72, 'T12 · Variables derivadas (solo con información pasada, sin cruzar bloques no aptos)', fontsize=9.5, color=COLOR_ORANGE, fontweight='bold')
pasos = ['T12.1 Continuidad\n(apto_modelado)', 'T12.2 Rezagos\nt-1, 2, 3, 6, 12, 24', 'T12.3 Media y desv.\nmóvil 6 h y 24 h',
         'T12.4 Tasas de cambio\n(3 h y 6 h)', 'T12.5 % de la ventana\nimputada']
for i, t in enumerate(pasos):
    x = 0.2 + i * 2.6
    caja(ax, x, 2.35, 2.35, 1.1, t, color=COLOR_ORANGE, fontsize=8.4)
    if i < 4: flecha(ax, (x + 2.35, 2.9), (x + 2.6, 2.9), color=COLOR_ORANGE)

ax.text(0.2, 1.97, 'T11 (parte 2) · Partición temporal y normalización sin fuga de datos', fontsize=9.5, color=COLOR_NAVY, fontweight='bold')
caja(ax, 0.2, 0.75, 3.7, 1.0, 'Partición cronológica 70/15/15\n+ purga de 24 h + folds walk-forward', color=COLOR_NAVY, fontsize=8.6)
flecha(ax, (3.9, 1.25), (4.4, 1.25))
caja(ax, 4.4, 0.75, 3.7, 1.0, 'Escalador ajustado SOLO con\nentrenamiento → aplicado a val. y prueba', color=COLOR_NAVY, fontsize=8.6)
flecha(ax, (8.1, 1.25), (8.6, 1.25))
caja(ax, 8.6, 1.38, 4.4, 0.62, 'Clasificador: todas las filas entrenables  →  ocurre_evento_24h', color=COLOR_TEAL, fontsize=7.8)
caja(ax, 8.6, 0.62, 4.4, 0.62, 'Regresor doble salida: filas con evento  →  inicio y duración', color=COLOR_ORANGE, fontsize=7.8)
ax.text(0.2, 0.2, 'Salida → HU07 / HU08: dataset_features_HU06.csv + escalador + catálogo de variables + configuración', fontsize=9.2, color=COLOR_NAVY, fontweight='bold')
plt.tight_layout(); plt.show()

### Carga del insumo (salida de HU05)

Se lee `dataset_limpio_horario_HU05.csv` (malla horaria 2025-05-23 → 2026-08-12 con banderas `apto_modelado` y `n_predictores_imputados`, y los tres *targets* ya reconstruidos con los eventos limpios). Se **valida el esquema** y que la malla siga siendo continua antes de transformar nada.

In [ ]:
# ==== Carga del insumo y validación de esquema ====
if not Path(RUTA_DATASET).exists():
    try:
        from google.colab import files
        print(f'Falta {RUTA_DATASET}. Si HU05 ya corrió en esta sesión, el archivo debería existir; si no, súbelo:')
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'No se encontró {RUTA_DATASET} en el directorio de trabajo.')

df_hu05 = pd.read_csv(RUTA_DATASET, index_col='fecha_hora', parse_dates=['fecha_hora'])
if df_hu05['apto_modelado'].dtype != bool:           # por si el CSV trae 'True'/'False' como texto
    df_hu05['apto_modelado'] = df_hu05['apto_modelado'].astype(str).str.lower().eq('true')

requeridas = VARS_CONTINUAS + [VAR_DIRECCION, 'apto_modelado', 'n_predictores_imputados'] + TARGETS
faltan = [c for c in requeridas if c not in df_hu05.columns]
assert not faltan, f'Faltan columnas en el dataset de HU05: {faltan}'
assert df_hu05.index.is_unique and df_hu05.index.is_monotonic_increasing, 'El índice debe ser único y ordenado'
assert (df_hu05.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all(), 'La malla horaria no es continua'

n_total = len(df_hu05)
print(f'Dataset HU05: {n_total:,} filas x {df_hu05.shape[1]} columnas  ({df_hu05.index.min()} -> {df_hu05.index.max()})')
print(f"Horas aptas para modelado (apto_modelado): {int(df_hu05['apto_modelado'].sum()):,} de {n_total:,} "
      f"({100*df_hu05['apto_modelado'].mean():.2f} %)")
print(f"Horas con evento en las próximas {HORIZONTE_H} h (clase positiva): {int(df_hu05[TARGET_CLF].sum()):,} "
      f"({100*df_hu05[TARGET_CLF].mean():.1f} %)")
display(df_hu05.head(3))

## 3.2.1 T11 (parte 1) — Seleccionar y codificar variables

### T11.1 Descartar redundancias y variables de diagnóstico

| Columna | Decisión | Motivo |
|---|---|---|
| `velocidad_viento_mph` | **Descartar** | Es la **misma variable** que `velocidad_viento_ms` en otra unidad (HU05 la reconstruye desde m/s). Dejar las dos duplicaría información y, en modelos lineales, generaría colinealidad perfecta. Se conserva **m/s**, la unidad usada en la literatura del Estado del Arte. |
| `temperatura_interna_datalogger_c`, `voltaje_datalogger_v` | **Descartar** | Son variables de **diagnóstico del equipo**, no condiciones meteo-oceanográficas. Sirvieron en HU05 para explicar vacíos compartidos, pero no deben alimentar el modelo (el voltaje podría correlacionar con eventos por causas ajenas al clima). |
| `n_predictores_imputados`, `apto_modelado` | **Control de calidad** | No son predictores: sirven para decidir qué filas son utilizables y qué proporción de cada ventana fue imputada. |
| `ocurre_evento_24h`, `horas_hasta_inicio_24h`, `duracion_indisponibilidad_h` | **Objetivos** | Nunca entran en `X`. |

In [ ]:
# ==== T11.1 - Selección de columnas ====
def _decision_columna(c):
    if c in COLUMNAS_DESCARTADAS:                 return 'DESCARTAR', COLUMNAS_DESCARTADAS[c]
    if c in VARS_CONTINUAS:                       return 'CONSERVAR (continua)', 'Predictor meteo-oceanográfico; se normaliza con el escalador de entrenamiento'
    if c == VAR_DIRECCION:                        return 'CODIFICAR (seno/coseno)', 'Variable circular: 0° y 360° son la misma dirección'
    if c in TARGETS:                              return 'OBJETIVO', 'Variable a predecir (nunca entra en X)'
    if c in ('apto_modelado', 'n_predictores_imputados'): return 'CONTROL DE CALIDAD', 'Define filas utilizables y proporción imputada'
    return 'REVISAR', 'Columna no prevista: decidir si es predictor'

decision_columnas = pd.DataFrame([(c, *_decision_columna(c)) for c in df_hu05.columns],
                                 columns=['columna', 'decision', 'motivo'])
display(decision_columnas)
assert (decision_columnas['decision'] != 'REVISAR').all(), 'Hay columnas no previstas: revisar la tabla anterior'

# Evidencia de que mph y m/s son la misma variable
if 'velocidad_viento_mph' in df_hu05.columns:
    dif = (df_hu05['velocidad_viento_mph'] * 0.44704 - df_hu05['velocidad_viento_ms']).abs().max()
    print(f'Diferencia máxima |mph x 0.44704 - m/s| = {dif:.2e}  ->  misma variable en distinta unidad; se conserva m/s.')

### T11.2 Dirección del viento como seno y coseno

La dirección es una variable **circular**: 359° y 1° están a solo 2° de distancia, pero como número difieren en 358. Un modelo que reciba grados crudos trataría ese par como opuestos. Se reemplaza por **dos componentes** que sí respetan la circularidad:

$$\text{dir\_viento\_sin} = \sin(\theta), \qquad \text{dir\_viento\_cos} = \cos(\theta), \qquad \theta = \text{grados}\times\frac{\pi}{180}$$

Cada hora queda como un punto sobre el círculo unitario (`sin² + cos² = 1`). Es coherente con HU05, donde la dirección ya se interpoló de forma circular. No requiere normalización posterior: ya está en [−1, 1].

In [ ]:
# ==== T11.2 - Codificación circular de la dirección del viento ====
_rad = np.deg2rad(df_hu05[VAR_DIRECCION])
base_hu06 = pd.DataFrame(index=df_hu05.index)
for v in VARS_CONTINUAS:
    base_hu06[v] = df_hu05[v]
base_hu06['dir_viento_sin'] = np.sin(_rad)
base_hu06['dir_viento_cos'] = np.cos(_rad)
VARS_CIRCULARES = ['dir_viento_sin', 'dir_viento_cos']
VARS_BASE = list(base_hu06.columns)

norma = np.sqrt(base_hu06['dir_viento_sin']**2 + base_hu06['dir_viento_cos']**2).dropna()
print(f'Verificación: sin² + cos² = 1 en todas las horas (desviación máx. = {(norma - 1).abs().max():.1e})')
ang_obs = df_hu05[VAR_DIRECCION].dropna()
print(f'Dirección observada: {ang_obs.min():.0f}° – {ang_obs.max():.0f}°  (la codificación igual queda lista para cualquier valor 0–360°)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.1, 1, 1]})
ax = axes[0]
ax.hist(ang_obs, bins=36, color=COLOR_GRAY, edgecolor='white')
ax.set_xlabel('direccion_viento_deg (grados crudos)'); ax.set_ylabel('horas')
ax.set_title('Antes: una sola columna en grados', fontsize=11, color=COLOR_NAVY, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
th = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(th), np.sin(th), color=COLOR_GRAY_LT, lw=6, zorder=1)
sc = ax.scatter(base_hu06['dir_viento_cos'], base_hu06['dir_viento_sin'], c=ang_obs.reindex(base_hu06.index),
                cmap='viridis', s=6, alpha=0.5, zorder=2)
ax.set_aspect('equal'); ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
ax.set_xlabel('dir_viento_cos'); ax.set_ylabel('dir_viento_sin')
ax.set_title('Después: puntos sobre el círculo', fontsize=11, color=COLOR_NAVY, fontweight='bold')
fig.colorbar(sc, ax=ax, shrink=0.75, label='grados')
ax.spines[['top', 'right']].set_visible(False)

ax = axes[2]
demo = np.array([350, 10, 180])
for ang, col, lab in zip(demo, [COLOR_TEAL, COLOR_ORANGE, COLOR_NAVY], ['350°', '10°', '180°']):
    ax.annotate('', xy=(np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))), xytext=(0, 0),
                arrowprops=dict(arrowstyle='-|>', color=col, lw=2.2))
    ax.text(1.18*np.cos(np.deg2rad(ang)), 1.18*np.sin(np.deg2rad(ang)), lab, color=col, ha='center', va='center', fontweight='bold')
ax.plot(np.cos(th), np.sin(th), color=COLOR_GRAY_LT, lw=3)
ax.set_aspect('equal'); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.axis('off')
ax.set_title('350° y 10° quedan cerca\n(como grados difieren en 340)', fontsize=11, color=COLOR_NAVY, fontweight='bold')
fig.suptitle('T11.2 — Codificación circular de la dirección del viento', fontsize=13, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

## 3.2.2 T12 — Generar variables derivadas meteo-oceanográficas

Un **rezago** (t-1, t-6, t-24…) es una variable derivada: no existe en el dato crudo, se construye a partir de las variables ya limpias. Las condiciones de mar y viento no afectan la operación de forma instantánea sino con un efecto **acumulado y desfasado**, por eso se incorpora memoria temporal explícita (TB2, inciso g).

### Familias de variables derivadas

| Familia | Definición | Variables base | Parámetros |
|---|---|---|---|
| **Valor actual** | Lectura de la hora de origen `t` | 5 continuas + `dir_viento_sin/cos` | — |
| **Rezagos** | `x(t-k)` | 5 continuas + `dir_viento_sin/cos` | k = 1, 2, 3, 6, 12, 24 h |
| **Media móvil** | media de `x(t-w+1) … x(t)` | 5 continuas + `dir_viento_sin/cos` (media circular) | w = 6 y 24 h |
| **Desviación móvil** | desviación estándar en la misma ventana | solo 5 continuas | w = 6 y 24 h |
| **Tasa de cambio** | `x(t) − x(t-k)` (tendencia: p. ej. caída de presión, crecimiento del oleaje) | solo 5 continuas | k = 3 y 6 h |
| **Proporción imputada** | fracción de los datos de la ventana que HU05 interpoló | global (usa `n_predictores_imputados`) | w = 6 y 24 h |

### Regla de continuidad: una ventana que cruza un bloque no apto no es un dato continuo

HU05 dejó marcadas las horas con `apto_modelado = False` (fallas prolongadas). Si un rezago o una ventana móvil **atraviesa** una de esas horas, mezclaría datos medidos antes y después de un hueco como si fueran vecinos. Regla aplicada:

1. Para cada hora `t` se calcula `racha_apto(t)`: cuántas horas aptas **consecutivas** terminan en `t`.
2. Un rezago de `k` horas (y una tasa de cambio de `k` horas) solo es válido si `racha_apto(t) ≥ k + 1`: **todas** las horas entre `t-k` y `t` son aptas.
3. Una ventana móvil de `w` horas solo es válida si `racha_apto(t) ≥ w`.
4. Cuando no se cumple, la variable queda en **NaN**; **no se rellena ni se interpola**. La fila no se usa para entrenar.

Consecuencia: tras cada bloque no apto se pierden las primeras 25 horas como filas entrenables (el rezago de 24 h necesita `t-24 … t`). Es un costo deliberado de no inventar continuidad.

### Regla de causalidad (anti *data-leakage*)

Toda variable en `t` usa **solo información de `t` hacia atrás**. Nunca se usan rezagos negativos ni ventanas centradas. El target de `t` mira hacia adelante (`t, t+24 h]`; los predictores, jamás.

In [ ]:
# ==== T12 - Construcción de variables derivadas (función reproducible) ====
def construir_features(base, apto, imputados, lags=None, ventanas=None, tasas=None, n_pred=N_PRED):
    """Construye las variables derivadas usando SOLO información de t y anteriores.
    Un rezago/ventana que cruza una hora no apta queda en NaN (no se rellena).
    Devuelve (X, meta_calidad, racha_apto, catalogo)."""
    lags     = LAGS_H if lags is None else lags
    ventanas = VENTANAS_H if ventanas is None else ventanas
    tasas    = TASAS_CAMBIO_H if tasas is None else tasas
    apto = apto.astype(bool)
    racha = apto.astype(int).groupby((~apto).cumsum()).cumsum()     # horas aptas consecutivas que terminan en t
    cols, cat = {}, []

    def agregar(nombre, serie, var, tipo, param):
        cols[nombre] = serie
        cat.append({'variable': nombre, 'variable_base': var, 'tipo': tipo, 'parametro_h': param,
                    'escalar': var not in VARS_CIRCULARES})

    for v in base.columns:                                           # 1) valor actual
        agregar(v, base[v].where(apto), v, 'actual', 0)
    for v in base.columns:                                           # 2) rezagos
        for k in lags:
            agregar(f'{v}_lag{k}h', base[v].shift(k).where(racha >= k + 1), v, 'rezago', k)
    for v in base.columns:                                           # 3) media y desviación móvil
        s = base[v].where(apto)
        for w in ventanas:
            ok = racha >= w
            agregar(f'{v}_media{w}h', s.rolling(w, min_periods=w).mean().where(ok), v, 'media_movil', w)
            if v not in VARS_CIRCULARES:
                agregar(f'{v}_std{w}h', s.rolling(w, min_periods=w).std().where(ok), v, 'desv_movil', w)
    for v in [c for c in base.columns if c not in VARS_CIRCULARES]:  # 4) tasas de cambio
        for k in tasas:
            agregar(f'{v}_delta{k}h', (base[v] - base[v].shift(k)).where(racha >= k + 1), v, 'tasa_cambio', k)

    meta = {}
    for w in ventanas:                                               # 5) proporción de la ventana que fue imputada
        meta[f'frac_imputada_{w}h'] = (imputados.rolling(w, min_periods=w).sum() / (n_pred * w)).where(racha >= w)
    return pd.DataFrame(cols), pd.DataFrame(meta), racha, pd.DataFrame(cat)

X_feat, meta_calidad, racha_apto, catalogo = construir_features(
    base_hu06, df_hu05['apto_modelado'], df_hu05['n_predictores_imputados'])

# Matriz de predictores (la proporción imputada es metadato salvo que se pida lo contrario)
X_modelo = X_feat.join(meta_calidad) if INCLUIR_FRAC_IMPUTADA_EN_X else X_feat
X_COLS = list(X_modelo.columns)
COLS_ESCALAR = [c for c in catalogo.loc[catalogo['escalar'], 'variable'] if c in X_COLS]
COLS_SIN_ESCALAR = [c for c in X_COLS if c not in COLS_ESCALAR]

print(f'Variables derivadas generadas: {X_feat.shape[1]}  ·  predictores en X: {len(X_COLS)}  ·  se normalizan: {len(COLS_ESCALAR)}  ·  no se normalizan (seno/coseno, % imputado): {len(COLS_SIN_ESCALAR)}')
resumen_tipos = (catalogo.groupby('tipo', sort=False)
                 .agg(n_variables=('variable', 'size'), parametros_h=('parametro_h', lambda s: ', '.join(map(str, sorted(set(s))))))
                 .reset_index())
display(resumen_tipos)
display(catalogo.head(8))

In [ ]:
# ==== T12 - Filas entrenables (continuidad + proporción imputada) y los dos conjuntos de tareas ====
feat_completa  = X_feat.notna().all(axis=1)                          # todos los rezagos/ventanas válidos
frac_max_col   = f'frac_imputada_{max(VENTANAS_H)}h'
frac_ok        = meta_calidad[frac_max_col] <= MAX_FRAC_IMPUTADA_VENTANA

entrenable_clf = feat_completa & frac_ok                              # clasificador: toda fila con features completas
entrenable_reg = (entrenable_clf & df_hu05[TARGET_CLF].eq(1)
                  & df_hu05[TARGETS_REG].notna().all(axis=1))         # regresor: solo orígenes con evento en las próximas 24 h

assert (feat_completa == (racha_apto >= MIN_RACHA_H)).all(), 'La completitud de features no coincide con la regla de continuidad'

embudo_filas = [
    ('Horas en la malla (HU05)', len(df_hu05)),
    ('Aptas para modelado (apto_modelado)', int(df_hu05['apto_modelado'].sum())),
    (f'Con {MIN_RACHA_H} h aptas consecutivas (features completas)', int(feat_completa.sum())),
    (f'Ventana de 24 h con ≤ {MAX_FRAC_IMPUTADA_VENTANA:.0%} imputado  →  CLASIFICADOR', int(entrenable_clf.sum())),
    ('Con evento en las próximas 24 h  →  REGRESOR doble salida', int(entrenable_reg.sum())),
]
tabla_embudo = pd.DataFrame(embudo_filas, columns=['Etapa', 'filas'])
tabla_embudo['% de la malla'] = (100 * tabla_embudo['filas'] / len(df_hu05)).round(2)
display(tabla_embudo)

etq, val = zip(*embudo_filas)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={'width_ratios': [1.45, 1]})
ax = axes[0]
y = np.arange(len(etq))
ax.barh(y, val, color=[COLOR_GRAY, COLOR_GRAY, COLOR_TEAL, COLOR_NAVY, COLOR_ORANGE])
for yi, v in zip(y, val): ax.text(v, yi, f' {v:,}', va='center', fontsize=9, color=COLOR_NAVY, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(etq, fontsize=8.6); ax.invert_yaxis(); ax.set_xlim(0, max(val) * 1.15)
ax.spines[['top', 'right']].set_visible(False); ax.set_xlabel('N.° de horas (filas)')
ax.set_title('De la malla horaria a las filas entrenables', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')

ax = axes[1]
mensual = pd.DataFrame({'entrenable': entrenable_clf, 'apto': df_hu05['apto_modelado']}).resample('MS').mean() * 100
ax.bar(mensual.index, mensual['apto'], width=24, color=COLOR_GRAY_LT, label='apto_modelado')
ax.bar(mensual.index, mensual['entrenable'], width=24, color=COLOR_TEAL, label='fila entrenable')
ax.set_ylim(80, 101); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')); ax.tick_params(axis='x', labelsize=8, rotation=30)
ax.set_ylabel('% de horas del mes'); ax.legend(frameon=False, fontsize=8.5, loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2)
ax.spines[['top', 'right']].set_visible(False)
ax.set_title('Pérdida por continuidad, mes a mes', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')
fig.suptitle('T12 — Cuántas filas quedan después de exigir ventanas continuas', fontsize=13, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

fi = meta_calidad[frac_max_col].dropna()
print(f'Proporción imputada de la ventana de {max(VENTANAS_H)} h: media {100*fi.mean():.2f} %, máx. {100*fi.max():.1f} %  ·  '
      f'filas por encima del umbral de {MAX_FRAC_IMPUTADA_VENTANA:.0%}: {int((fi > MAX_FRAC_IMPUTADA_VENTANA).sum())}')

In [ ]:
# ==== Vista visual - Un bloque no apto y qué variables derivadas invalida ====
no_apto = ~df_hu05['apto_modelado']
grupo = (no_apto != no_apto.shift(fill_value=False)).cumsum()
bloques_hu06 = pd.DataFrame([{'inicio': b.index[0], 'fin': b.index[-1], 'horas': len(b)}
                             for _, b in df_hu05[no_apto].groupby(grupo[no_apto])])
print(f'Bloques no aptos heredados de HU05: {len(bloques_hu06)}  ·  horas no aptas: {int(no_apto.sum())}  ·  bloque más largo: {int(bloques_hu06["horas"].max())} h')

# Se elige el bloque más largo que deje margen a ambos lados
cand = bloques_hu06[(bloques_hu06['inicio'] > df_hu05.index[0] + pd.Timedelta(hours=40)) &
                    (bloques_hu06['fin'] < df_hu05.index[-1] - pd.Timedelta(hours=60))]
b = cand.sort_values('horas', ascending=False).iloc[0]
ini_v, fin_v = b['inicio'] - pd.Timedelta(hours=36), b['fin'] + pd.Timedelta(hours=60)
ventana = df_hu05.loc[ini_v:fin_v]

v_demo = 'altura_olas_m'
feats_demo = [f'{v_demo}_lag1h', f'{v_demo}_lag6h', f'{v_demo}_lag24h', f'{v_demo}_media6h', f'{v_demo}_media24h']
M = X_feat.loc[ini_v:fin_v, feats_demo].notna().T.values.astype(int)

from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(2, 1, figsize=(14, 6.2), sharex=True, gridspec_kw={'height_ratios': [1.3, 1]})
ax = axes[0]
ax.plot(ventana.index, ventana[v_demo], color=COLOR_NAVY, lw=1.6, marker='o', ms=3)
ax.axvspan(b['inicio'], b['fin'] + pd.Timedelta(hours=1), color=COLOR_GOLD, alpha=0.35, lw=0, label=f'bloque no apto ({int(b["horas"])} h)')
ax.set_ylabel(v_demo); ax.legend(frameon=False, fontsize=9, loc='upper right'); ax.spines[['top', 'right']].set_visible(False)
ax.set_title(f'Bloque no apto del {b["inicio"]:%d-%b-%Y %H:%M} y las variables derivadas que invalida', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')

ax = axes[1]
x0 = mdates.date2num(ventana.index[0]); x1 = mdates.date2num(ventana.index[-1]) + 1/24
ax.imshow(M, aspect='auto', cmap=ListedColormap([COLOR_RED, COLOR_TEAL]), vmin=0, vmax=1,
          extent=[x0, x1, len(feats_demo), 0], interpolation='nearest')
ax.set_yticks(np.arange(len(feats_demo)) + 0.5); ax.set_yticklabels([f.replace(f'{v_demo}_', '') for f in feats_demo], fontsize=9)
ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b %Hh')); ax.tick_params(axis='x', labelsize=8, rotation=20)
ax.legend(handles=[Patch(color=COLOR_TEAL, label='variable válida'), Patch(color=COLOR_RED, label='NaN (cruza el bloque o no hay historia suficiente)')],
          frameon=False, fontsize=8.5, loc='upper right', bbox_to_anchor=(1.0, -0.22), ncol=2)
fig.tight_layout(); plt.show()
print('Lectura: tras el bloque, cada variable derivada recupera validez recién cuando toda su ventana vuelve a ser apta '
      '(lag1 casi de inmediato; lag24 y media24h, 24-25 h después).')

In [ ]:
# ==== Vista visual - Cómo se ven las variables derivadas antes de los eventos ====
inicio_ev = (pd.Series(df_hu05.index + pd.to_timedelta(df_hu05['horas_hasta_inicio_24h'], unit='h'))
             .dropna().dt.round('min').drop_duplicates())
ini_w = pd.Timestamp('2025-08-10'); fin_w = ini_w + pd.Timedelta(days=7)
fig, axes = plt.subplots(3, 1, figsize=(14, 7.6), sharex=True)
for ax, v, etiqueta in zip(axes, ['altura_olas_m', 'velocidad_viento_ms', 'presion_atmosferica_hpa'],
                           ['Altura de olas (m)', 'Velocidad del viento (m/s)', 'Presión (hPa)']):
    ax.plot(base_hu06.loc[ini_w:fin_w, v], color=COLOR_GRAY, lw=1.0, label='valor actual')
    ax.plot(X_feat.loc[ini_w:fin_w, f'{v}_media6h'], color=COLOR_TEAL, lw=1.8, label='media móvil 6 h')
    ax.plot(X_feat.loc[ini_w:fin_w, f'{v}_media24h'], color=COLOR_NAVY, lw=2.0, label='media móvil 24 h')
    for t in inicio_ev[(inicio_ev >= ini_w) & (inicio_ev <= fin_w)]:
        ax.axvline(t, color=COLOR_RED, ls='--', lw=1.1, alpha=0.8)
    ax.set_ylabel(etiqueta, fontsize=9); ax.spines[['top', 'right']].set_visible(False)
axes[0].plot([], [], color=COLOR_RED, ls='--', label='inicio de evento de indisponibilidad')
axes[0].legend(frameon=False, fontsize=8.5, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.28))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b')); axes[-1].xaxis.set_major_locator(mdates.DayLocator())
fig.suptitle('T12 — Valor actual vs. medias móviles (7 días de ejemplo)', fontsize=13, color=COLOR_NAVY, fontweight='bold', y=1.0)
fig.tight_layout(); plt.show()

In [ ]:
# ==== Resumen del catálogo de variables ====
fig, ax = plt.subplots(figsize=(9, 3.4))
orden = ['actual', 'rezago', 'media_movil', 'desv_movil', 'tasa_cambio']
cnt = catalogo['tipo'].value_counts().reindex(orden).dropna()
colores = [COLOR_GRAY, COLOR_TEAL, COLOR_NAVY, COLOR_GOLD, COLOR_ORANGE]
ax.barh(cnt.index, cnt.values, color=colores[:len(cnt)])
for yi, v in enumerate(cnt.values): ax.text(v, yi, f' {int(v)}', va='center', fontsize=9.5, fontweight='bold', color=COLOR_NAVY)
ax.invert_yaxis(); ax.set_xlim(0, cnt.max() * 1.15); ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel('N.° de variables'); ax.set_title(f'Predictores por familia (total en X: {len(X_COLS)})', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

## 3.2.3 T11 (parte 2) — Partición temporal y normalización sin fuga de datos

### T11.3 Partición cronológica con purga

Los datos son una serie de tiempo: **no se mezclan al azar**. Se parte en bloques consecutivos (TB2, inciso h):

| Bloque | Proporción | Uso previsto |
|---|---|---|
| Entrenamiento | 70 % más antiguo | Ajustar el escalador y los modelos |
| Validación | siguiente 15 % | Elegir hiperparámetros y comparar algoritmos |
| Prueba | último 15 % | Evaluación final, **no se toca** hasta HU09 |

**Purga de 24 h antes de cada corte.** La etiqueta de un origen `t` mira hasta `t + 24 h`. Si `t` está a menos de 24 h del corte, su etiqueta depende de eventos que ocurren del otro lado. Esas horas se marcan `purga` y no se usan en ningún conjunto.

Adicionalmente se definen **4 ventanas walk-forward** de entrenamiento creciente sobre el tramo entrenamiento + validación, para medir la **estabilidad** del desempeño entre periodos (criterio de selección del TB2 y de HU08). La prueba final queda fuera de los *folds*.

In [ ]:
# ==== T11.3 - Partición cronológica con purga ====
idx = df_hu05.index
n = len(idx)
corte_train = idx[int(n * FRAC_TRAIN)]
corte_val   = idx[int(n * (FRAC_TRAIN + FRAC_VAL))]
purga = pd.Timedelta(hours=GAP_H)

particion = pd.Series('prueba', index=idx, name='particion')
particion[idx < corte_val] = 'validacion'
particion[idx < corte_train] = 'entrenamiento'
particion[(idx >= corte_train - purga) & (idx < corte_train)] = 'purga'
particion[(idx >= corte_val - purga) & (idx < corte_val)] = 'purga'
ORDEN_PART = ['entrenamiento', 'purga', 'validacion', 'prueba']

filas = []
tramo = (particion != particion.shift()).cumsum()            # tramos contiguos (la purga aparece dos veces: antes de validación y antes de prueba)
for g in tramo.unique():
    m = tramo == g
    p = particion[m].iloc[0]
    mc, mr = m & entrenable_clf, m & entrenable_reg
    filas.append({'particion': p, 'desde': idx[m].min(), 'hasta': idx[m].max(), 'horas': int(m.sum()),
                  'filas_clasificador': int(mc.sum()),
                  '% con evento': round(100 * df_hu05.loc[mc, TARGET_CLF].mean(), 1) if mc.any() else np.nan,
                  'filas_regresor': int(mr.sum())})
resumen_particion = pd.DataFrame(filas)
display(resumen_particion)

pos_prueba = int(df_hu05.loc[(particion == 'prueba') & entrenable_clf, TARGET_CLF].sum())
print(f'Cortes: entrenamiento < {corte_train}  ·  validación < {corte_val}  ·  prueba desde {corte_val}')
print(f'Positivos en prueba: {pos_prueba}  ({"suficientes para F1/AUC" if pos_prueba >= 100 else "POCOS: F1/AUC serán inestables"})')
print('Nota: la proporción de horas con evento cambia entre estaciones (más eventos en invierno austral); por eso se reporta por partición y se mide la estabilidad entre ventanas.')

In [ ]:
# ==== T11.3 - Ventanas walk-forward (entrenamiento creciente) ====
def construir_folds(indice, corte_final, n_folds, gap_h):
    """Divide el tramo previo a la prueba en n_folds+1 bloques consecutivos. El fold i entrena con los bloques 1..i
    y valida con el bloque i+1; se purgan gap_h+1 horas entre entrenamiento y validación."""
    region = indice[indice < corte_final - pd.Timedelta(hours=gap_h)]
    lim = np.linspace(0, len(region), n_folds + 2).astype(int)
    folds = []
    for i in range(n_folds):
        v_ini, v_fin = region[lim[i + 1]], region[lim[i + 2] - 1]
        folds.append({'fold': i + 1, 'train_ini': region[0], 'train_fin': v_ini - pd.Timedelta(hours=gap_h + 1),
                      'val_ini': v_ini, 'val_fin': v_fin})
    return folds

FOLDS = construir_folds(idx, corte_val, N_FOLDS_WF, GAP_H)

def mascara_fold(f, parte):
    ini, fin = (f['train_ini'], f['train_fin']) if parte == 'train' else (f['val_ini'], f['val_fin'])
    return pd.Series((idx >= ini) & (idx <= fin), index=idx)

filas = []
for f in FOLDS:
    mt, mv = mascara_fold(f, 'train'), mascara_fold(f, 'val')
    filas.append({'fold': f['fold'], 'entrenamiento': f"{f['train_ini']:%d-%b-%y} -> {f['train_fin']:%d-%b-%y}",
                  'validacion': f"{f['val_ini']:%d-%b-%y} -> {f['val_fin']:%d-%b-%y}",
                  'filas_clf_train': int((mt & entrenable_clf).sum()), 'filas_clf_val': int((mv & entrenable_clf).sum()),
                  '% con evento (val)': round(100 * df_hu05.loc[mv & entrenable_clf, TARGET_CLF].mean(), 1),
                  'filas_reg_train': int((mt & entrenable_reg).sum()), 'filas_reg_val': int((mv & entrenable_reg).sum())})
tabla_folds = pd.DataFrame(filas)
display(tabla_folds)

# ---- Vista visual: línea de tiempo de la partición y de los folds
fig, axes = plt.subplots(2, 1, figsize=(14, 6.4), sharex=True, gridspec_kw={'height_ratios': [1.35, 1]})
colores_part = {'entrenamiento': COLOR_TEAL_LT, 'purga': COLOR_GRAY, 'validacion': COLOR_GOLD, 'prueba': COLOR_ORANGE}
ax = axes[0]
for g in tramo.unique():                                   # un bloque por tramo contiguo (la purga son 2 franjas de 24 h)
    m = tramo == g
    p = particion[m].iloc[0]
    ax.axvspan(idx[m].min(), idx[m].max() + pd.Timedelta(hours=1), color=colores_part[p], alpha=0.55 if p != 'purga' else 1.0, lw=0)
semanal = df_hu05[TARGET_CLF].resample('W').mean() * 100
ax.plot(semanal.index, semanal.values, color=COLOR_NAVY, lw=1.8, marker='o', ms=3)
ax.set_ylabel('% de horas con evento\nen las próximas 24 h (semanal)', fontsize=9); ax.set_ylim(0, 100)
ax.legend(handles=[Patch(color=colores_part[p], label=p) for p in ORDEN_PART], frameon=False, fontsize=8.5, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.2))
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
for i, f in enumerate(FOLDS):
    y0 = len(FOLDS) - i - 0.8
    ax.barh(y0 + 0.4, mdates.date2num(f['train_fin']) - mdates.date2num(f['train_ini']), left=mdates.date2num(f['train_ini']),
            height=0.6, color=COLOR_TEAL)
    ax.barh(y0 + 0.4, mdates.date2num(f['val_fin']) - mdates.date2num(f['val_ini']), left=mdates.date2num(f['val_ini']),
            height=0.6, color=COLOR_GOLD)
    ax.text(mdates.date2num(idx[0]), y0 + 0.4, f"fold {f['fold']} ", ha='right', va='center', fontsize=8.5, color=COLOR_NAVY)
ax.barh(-0.4, mdates.date2num(idx[-1]) - mdates.date2num(corte_val), left=mdates.date2num(corte_val), height=0.6, color=COLOR_ORANGE)
ax.text(mdates.date2num(idx[0]), -0.4, 'prueba final ', ha='right', va='center', fontsize=8.5, color=COLOR_NAVY)
ax.set_yticks([]); ax.set_ylim(-1, len(FOLDS) + 0.2)
ax.legend(handles=[Patch(color=COLOR_TEAL, label='entrenamiento'), Patch(color=COLOR_GOLD, label='validación'), Patch(color=COLOR_ORANGE, label='prueba (intocable hasta HU09)')],
          frameon=False, fontsize=8.5, ncol=3, loc='lower center', bbox_to_anchor=(0.5, -0.45))
ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')); ax.spines[['top', 'right', 'left']].set_visible(False)
fig.suptitle('T11.3 — Partición cronológica y ventanas walk-forward', fontsize=13, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

### T11.4 Normalización ajustada solo con entrenamiento

El escalador aprende estadísticos (media y desviación para `standard`). Si aprende con todo el histórico, la media de **prueba** ya influye en cómo se transforma el **entrenamiento**: el modelo "ve" el futuro sin querer.

**Procedimiento**

1. `fit` del escalador **solo** con las filas entrenables de la partición `entrenamiento`.
2. `transform` (sin volver a ajustar) sobre entrenamiento, validación y prueba.
3. **No se normalizan** `dir_viento_sin/cos` (ya están en [−1, 1]) ni la proporción imputada (ya está en [0, 1]).
4. **Los targets no se normalizan**: los regresores predicen **horas** directamente (MAE y RMSE se interpretan en horas).
5. Dentro de cada *fold* walk-forward el escalador se **vuelve a ajustar** con el entrenamiento de ese fold (`obtener_fold`).

> Los modelos basados en árboles (RF, GBM, XGBoost, LightGBM) no necesitan normalización, pero la **regresión lineal** del *benchmark* sí. Se entregan ambas versiones (`escalado=True/False`).

In [ ]:
# ==== T11.4 - Ajuste del escalador SOLO con entrenamiento ====
def crear_escalador(nombre=ESCALADOR):
    return {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}[nombre]

def aplicar_escalador(esc, X):
    Xs = X.copy()
    Xs[COLS_ESCALAR] = esc.transform(X[COLS_ESCALAR])          # transform: nunca vuelve a ajustar
    return Xs

mask_fit = (particion == 'entrenamiento') & entrenable_clf
escalador = crear_escalador().fit(X_modelo.loc[mask_fit, COLS_ESCALAR])
X_esc = aplicar_escalador(escalador, X_modelo)
print(f'Escalador "{ESCALADOR}" ajustado con {int(mask_fit.sum()):,} filas de ENTRENAMIENTO '
      f'({idx[mask_fit].min():%Y-%m-%d} -> {idx[mask_fit].max():%Y-%m-%d}) y aplicado a {len(COLS_ESCALAR)} variables.')

# Evidencia de por qué importa: lo que pasaría si se ajustara con todo el histórico (SOLO comparación, no se usa)
esc_global = crear_escalador().fit(X_modelo.loc[entrenable_clf, COLS_ESCALAR])
vars_ref = VARS_CONTINUAS
i_ref = [COLS_ESCALAR.index(v) for v in vars_ref]
if ESCALADOR == 'standard':
    comp = pd.DataFrame({'media_entrenamiento': escalador.mean_[i_ref], 'media_si_se_usara_todo': esc_global.mean_[i_ref],
                         'desv_entrenamiento': escalador.scale_[i_ref]}, index=vars_ref)
    comp['sesgo_en_desviaciones'] = (comp['media_si_se_usara_todo'] - comp['media_entrenamiento']) / comp['desv_entrenamiento']
    display(comp.round(3))
    print('Lectura: "sesgo_en_desviaciones" es cuánto se movería el cero de cada variable si el escalador hubiera visto validación y prueba. '
          'Cualquier valor distinto de 0 es información del futuro filtrada al entrenamiento.')

In [ ]:
# ==== Vista visual - Distribuciones por partición, antes y después de normalizar ====
vars_fig = ['altura_olas_m', 'velocidad_viento_ms', 'presion_atmosferica_hpa']
colores_p = {'entrenamiento': COLOR_TEAL, 'validacion': COLOR_GOLD, 'prueba': COLOR_ORANGE}
fig, axes = plt.subplots(2, 3, figsize=(15, 6.2))
for j, v in enumerate(vars_fig):
    for fila, (Xs, titulo) in enumerate([(X_modelo, 'unidades originales'), (X_esc, 'normalizada (escalador de entrenamiento)')]):
        ax = axes[fila, j]
        for p, col in colores_p.items():
            vals = Xs.loc[(particion == p) & entrenable_clf, v]
            ax.hist(vals, bins=30, density=True, histtype='step', lw=2, color=col, label=p)
        ax.set_title(f'{v}\n{titulo}', fontsize=9.5, color=COLOR_NAVY, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
        if j == 0 and fila == 0: ax.legend(frameon=False, fontsize=8.5)
fig.suptitle('T11.4 — La normalización se aprende en entrenamiento; validación y prueba conservan su propia distribución',
             fontsize=12.5, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()
print('Lectura: si validación o prueba se desplazan respecto del entrenamiento, el modelo verá valores fuera de lo aprendido '
      '(deriva estacional). Eso es información real del problema, y por eso no se "corrige" ajustando con todo el histórico.')

### Entrega para HU07 / HU08: dos tablas de entrenamiento, un solo dataset

Las funciones siguientes devuelven exactamente lo que necesitan los dos modelos del TB2. HU07 y HU08 no deben reconstruir nada: solo llamarlas.

| Función | Devuelve |
|---|---|
| `obtener_datos(conjunto, tarea, escalado)` | `X, y` de `entrenamiento` / `validacion` / `prueba`. `tarea='clasificacion'` → `y = ocurre_evento_24h`. `tarea='regresion'` → `y` con **dos columnas** (`horas_hasta_inicio_24h`, `duracion_indisponibilidad_h`) solo en orígenes con evento |
| `obtener_fold(k, tarea, escalado)` | `X_train, y_train, X_val, y_val` del *fold* `k` (1…4), con el escalador **reajustado** con el entrenamiento de ese *fold* |

In [ ]:
# ==== API para HU07/HU08 ====
def _y(tarea):
    if tarea == 'clasificacion': return df_hu05[TARGET_CLF].astype(int)
    if tarea == 'regresion':     return df_hu05[TARGETS_REG]
    raise ValueError("tarea debe ser 'clasificacion' o 'regresion'")

def _elegibles(tarea):
    return entrenable_clf if tarea == 'clasificacion' else entrenable_reg

def obtener_datos(conjunto, tarea, escalado=True):
    m = (particion == conjunto) & _elegibles(tarea)
    Xs = X_esc if escalado else X_modelo
    return Xs.loc[m, X_COLS], _y(tarea).loc[m]

def obtener_fold(k, tarea, escalado=True):
    f = FOLDS[k - 1]
    mt, mv = mascara_fold(f, 'train'), mascara_fold(f, 'val')
    if escalado:
        esc = crear_escalador().fit(X_modelo.loc[mt & entrenable_clf, COLS_ESCALAR])   # ajuste SOLO con el train del fold
        Xs = aplicar_escalador(esc, X_modelo)
    else:
        Xs = X_modelo
    mt, mv = mt & _elegibles(tarea), mv & _elegibles(tarea)
    y = _y(tarea)
    return Xs.loc[mt, X_COLS], y.loc[mt], Xs.loc[mv, X_COLS], y.loc[mv]

filas = []
for p in ['entrenamiento', 'validacion', 'prueba']:
    Xc, yc = obtener_datos(p, 'clasificacion'); Xr, yr = obtener_datos(p, 'regresion')
    filas.append({'particion': p, 'X_clasificador': Xc.shape, 'positivos': int(yc.sum()), 'X_regresor': Xr.shape, 'y_regresor': yr.shape})
display(pd.DataFrame(filas))

## Verificaciones finales de HU06

Se ejecutan **comprobaciones automáticas** de las reglas de este notebook. Las dos más importantes son pruebas de **fuga de datos**:

- **Prueba de causalidad:** se alteran los datos a partir de una hora `p` y se reconstruyen todas las variables; **ninguna fila anterior a `p` debe cambiar**. Si cambiara, esa variable estaría usando información del futuro.
- **Prueba del escalador:** el escalador debe haber visto **exactamente** las filas de entrenamiento y reproducir su media y desviación.

In [ ]:
# ==== Verificaciones finales (assertions) ====
# --- Prueba de causalidad: alterar el futuro a partir de p no debe cambiar ninguna fila anterior
p = len(df_hu05) // 2
base_pert = base_hu06.copy()
base_pert.iloc[p:] = base_pert.iloc[p:] * 1.37 + 0.5
imp_pert = df_hu05['n_predictores_imputados'].copy(); imp_pert.iloc[p:] = 6
X_pert, meta_pert, _, _ = construir_features(base_pert, df_hu05['apto_modelado'], imp_pert)
causal_ok = X_feat.iloc[:p].equals(X_pert.iloc[:p]) and meta_calidad.iloc[:p].equals(meta_pert.iloc[:p])
sensible_ok = not X_feat.iloc[p:p + 30].fillna(-999).equals(X_pert.iloc[p:p + 30].fillna(-999))   # la prueba no es vacía

# --- Reconstrucción manual de rezagos y ventanas en filas entrenables al azar
rng = np.random.default_rng(SEMILLA)
muestra = rng.choice(np.where(entrenable_clf.values)[0], size=min(300, int(entrenable_clf.sum())), replace=False)
ok_manual = True
for i in muestra:
    t = idx[i]
    for v in ['altura_olas_m', 'presion_atmosferica_hpa']:
        ok_manual &= np.isclose(X_feat.at[t, f'{v}_lag6h'], base_hu06.at[idx[i - 6], v])
        ok_manual &= np.isclose(X_feat.at[t, f'{v}_media24h'], base_hu06[v].iloc[i - 23:i + 1].mean())
        ok_manual &= np.isclose(X_feat.at[t, f'{v}_std6h'], base_hu06[v].iloc[i - 5:i + 1].std())
        ok_manual &= np.isclose(X_feat.at[t, f'{v}_delta3h'], base_hu06.at[t, v] - base_hu06.at[idx[i - 3], v])

# --- Ninguna fila entrenable tiene una hora no apta en sus 25 h de historia
horas_no_aptas_hist = (~df_hu05['apto_modelado']).astype(int).rolling(MIN_RACHA_H, min_periods=1).sum()

# --- Particiones
def _rango(p):
    m = particion == p
    return idx[m].min(), idx[m].max()
(tr0, tr1), (va0, va1), (te0, te1) = _rango('entrenamiento'), _rango('validacion'), _rango('prueba')

# --- Escalador
Xtr_fit = X_modelo.loc[mask_fit, COLS_ESCALAR]
Xtr_esc = X_esc.loc[mask_fit, COLS_ESCALAR]
reprod = X_modelo.copy()[COLS_SIN_ESCALAR].equals(X_esc[COLS_SIN_ESCALAR]) if COLS_SIN_ESCALAR else True

# --- Reproducibilidad: reconstruir da exactamente lo mismo
X_rep, meta_rep, _, _ = construir_features(base_hu06, df_hu05['apto_modelado'], df_hu05['n_predictores_imputados'])

# --- Folds
folds_ok = all(f['train_fin'] + pd.Timedelta(hours=GAP_H) < f['val_ini'] for f in FOLDS) and \
           all(FOLDS[i]['val_fin'] < FOLDS[i + 1]['val_ini'] for i in range(len(FOLDS) - 1)) and FOLDS[-1]['val_fin'] < corte_val
Xf_tr, yf_tr, Xf_va, yf_va = obtener_fold(1, 'clasificacion')

chequeos = {
    'Entradas: columnas descartadas ausentes de X': not any(c.startswith(('velocidad_viento_mph', 'temperatura_interna_datalogger', 'voltaje_datalogger')) for c in X_COLS),
    'Entradas: ningún target ni columna *_durante_evento_* en X': not any(c in TARGETS or 'durante_evento' in c for c in X_COLS),
    'Entradas: velocidad_viento_ms presente y mph descartada (una sola variable de viento)': 'velocidad_viento_ms' in X_COLS and not any('mph' in c for c in X_COLS),
    'Codificación: sin² + cos² = 1 en la dirección del viento': bool(((norma - 1).abs().max()) < 1e-9),
    'Codificación: dirección en grados no está en X': VAR_DIRECCION not in X_COLS,
    'Rezagos: se generan los 6 rezagos exigidos por el TB2 para cada variable base': all(f'{v}_lag{k}h' in X_COLS for v in VARS_BASE for k in [1, 2, 3, 6, 12, 24]),
    'Ventanas: media (y desviación en continuas) de 6 y 24 h': all(f'{v}_media{w}h' in X_COLS for v in VARS_BASE for w in [6, 24]) and all(f'{v}_std{w}h' in X_COLS for v in VARS_CONTINUAS for w in [6, 24]),
    'Continuidad: filas entrenables solo con racha apta >= mínima': bool((racha_apto[entrenable_clf] >= MIN_RACHA_H).all()),
    'Continuidad: ninguna fila entrenable cruza una hora no apta': bool((horas_no_aptas_hist[entrenable_clf] == 0).all()),
    'Continuidad: filas entrenables sin NaN en X': bool(X_modelo.loc[entrenable_clf, X_COLS].notna().all().all()),
    'Causalidad: alterar el futuro NO cambia ninguna fila anterior (sin fuga)': bool(causal_ok and sensible_ok),
    'Exactitud: rezagos/medias/desviaciones/tasas coinciden con el cálculo manual (300 filas)': bool(ok_manual),
    'Reproducibilidad: reconstruir las variables da exactamente el mismo resultado': bool(X_feat.equals(X_rep) and meta_calidad.equals(meta_rep)),
    'Targets: regresor solo con evento y targets no nulos en (0, 24]': bool(df_hu05.loc[entrenable_reg, 'horas_hasta_inicio_24h'].between(0, HORIZONTE_H, inclusive='right').all() and df_hu05.loc[entrenable_reg, TARGETS_REG].notna().all().all()),
    'Targets: sin 0 forzado donde no hay evento (NaN) — regla Q10': bool(df_hu05.loc[df_hu05[TARGET_CLF] == 0, 'horas_hasta_inicio_24h'].isna().all()),
    'Partición: entrenamiento < validación < prueba, sin solape': bool(tr1 < va0 and va1 < te0),
    f'Partición: purga de {GAP_H} h entre bloques': bool(va0 - tr1 > purga and te0 - va1 > purga),
    'Escalador: ajustado exactamente con las filas de entrenamiento': bool(getattr(escalador, 'n_samples_seen_', len(Xtr_fit)) == len(Xtr_fit) and (particion[mask_fit] == 'entrenamiento').all()),
    'Escalador: entrenamiento normalizado con media 0 y desv. 1' if ESCALADOR == 'standard' else 'Escalador: aplicado sin NaN nuevos': bool((np.allclose(Xtr_esc.mean(), 0, atol=1e-8) and np.allclose(Xtr_esc.std(ddof=0), 1, atol=1e-8)) if ESCALADOR == 'standard' else Xtr_esc.notna().all().all()),
    'Escalador: no se normalizan seno/coseno ni la proporción imputada': bool(reprod),
    'Escalador: el del fold 1 se reajusta con su propio entrenamiento': bool(ESCALADOR != 'standard' or Xf_tr[COLS_ESCALAR].mean().abs().max() < 1e-8),
    'Folds: entrenamiento previo a validación con purga y validaciones consecutivas': bool(folds_ok),
    'API: obtener_datos devuelve y de dos columnas para el regresor': bool(obtener_datos('entrenamiento', 'regresion')[1].shape[1] == 2),
}
tabla_chequeos6 = pd.DataFrame({'Verificación': list(chequeos), 'Resultado': ['OK' if bool(v) else 'FALLA' for v in chequeos.values()]})
display(tabla_chequeos6)
assert (tabla_chequeos6['Resultado'] == 'OK').all(), 'Hay verificaciones que fallan: revisar la tabla anterior.'
print(f'\n{len(chequeos)} verificaciones OK.')

### Diagnóstico de señal: ¿las variables cambian *antes* del inicio de un evento?

Antes de entrenar conviene mirar si existe una pista temporal que los rezagos puedan capturar. Se alinean los **inicios de evento** (reconstruidos desde `horas_hasta_inicio_24h`) y se promedia la **anomalía** de cada variable (valor menos su media semanal, en desviaciones estándar) en las horas previas y posteriores. Si las condiciones empeoraran antes del evento, la curva subiría a la izquierda del cero.

> Es un diagnóstico **descriptivo** (usa una media centrada, que sería ilegal como predictor) y solo incluye eventos del tramo entrenamiento + validación: **no toca la prueba final**.

In [ ]:
# ==== Diagnóstico descriptivo: promedio compuesto alrededor del inicio de los eventos ====
ev_ini = (pd.Series(idx + pd.to_timedelta(df_hu05['horas_hasta_inicio_24h'], unit='h'))
          .dropna().dt.round('h').drop_duplicates())
ev_ini = ev_ini[(ev_ini < corte_val - purga) & (ev_ini > idx[0] + pd.Timedelta(hours=48))]
offsets = np.arange(-48, 25)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9), sharex=True)
resumen_diag = {}
for ax, v, etq in zip(axes, ['altura_olas_m', 'velocidad_viento_ms', 'presion_atmosferica_hpa'],
                      ['Altura de olas', 'Velocidad del viento', 'Presión']):
    s_v = df_hu05[v]
    anom = (s_v - s_v.rolling(24 * 7, center=True, min_periods=72).mean()) / s_v.std()
    medias, ee = [], []
    for h in offsets:
        vals = anom.reindex(ev_ini + pd.Timedelta(hours=int(h))).dropna()
        medias.append(vals.mean()); ee.append(vals.std() / np.sqrt(max(len(vals), 1)))
    medias, ee = np.array(medias), np.array(ee)
    ax.fill_between(offsets, medias - 1.96 * ee, medias + 1.96 * ee, color=COLOR_TEAL_LT)
    ax.plot(offsets, medias, color=COLOR_NAVY, lw=1.8)
    ax.axvline(0, color=COLOR_RED, ls='--', lw=1.2); ax.axhline(0, color=COLOR_GRAY, lw=0.8)
    ax.set_title(etq, fontsize=11, color=COLOR_NAVY, fontweight='bold'); ax.set_xlabel('horas respecto al inicio del evento')
    ax.spines[['top', 'right']].set_visible(False)
    previas = medias[(offsets >= -24) & (offsets <= -3)]
    resumen_diag[v] = {'anomalia_media_24h_previas': round(float(previas.mean()), 3), 'anomalia_en_hora_0': round(float(medias[offsets == 0][0]), 3)}
axes[0].set_ylabel('anomalía media (desv. estándar)')
fig.suptitle(f'Diagnóstico — comportamiento medio de las variables alrededor de {len(ev_ini)} inicios de evento (entrenamiento + validación)',
             fontsize=12, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()
display(pd.DataFrame(resumen_diag).T)
print('Lectura: banda = intervalo del 95 % de la media. Si las 24 h previas quedan dentro de la banda alrededor de 0, el estado '
      'de las variables en horas anteriores no anticipa el evento en promedio; si solo sobresale la hora 0, el evento coincide con un pico puntual.')

### Prueba de humo: ¿el dataset es entrenable y trae señal?

Criterio de HU06 (TB2): *dataset reproducible y apto para entrenamiento*. Se entrena un modelo mínimo por tarea y se mide en validación y en las 4 ventanas walk-forward.

> **Esto NO es el *benchmark* ni el entrenamiento de HU07/HU08.** No hay ajuste de hiperparámetros, ni los cuatro algoritmos, ni prueba final (la partición `prueba` no se toca). Sirve para dos cosas: (1) confirmar que las matrices tienen la forma correcta y ningún NaN rompe el entrenamiento; (2) **dar una primera señal honesta** de si las variables derivadas anticipan el evento mejor que una referencia trivial. Las cifras no se reportan como resultados del proyecto.

In [ ]:
# ==== Prueba de humo (NO es el benchmark de HU07/HU08) ====
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error

def _clf_minimo(Xtr, ytr, Xva, yva):
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=SEMILLA).fit(Xtr, ytr)
    p = m.predict_proba(Xva)[:, 1]
    return roc_auc_score(yva, p), f1_score(yva, p >= 0.5), f1_score(yva, np.ones(len(yva))), 100 * yva.mean()

filas_clf = []
Xc_tr, yc_tr = obtener_datos('entrenamiento', 'clasificacion', escalado=False)
Xc_va, yc_va = obtener_datos('validacion',    'clasificacion', escalado=False)
filas_clf.append(('Validación (partición)', *_clf_minimo(Xc_tr, yc_tr, Xc_va, yc_va)))
for k in range(1, N_FOLDS_WF + 1):
    a, b, c, d = obtener_fold(k, 'clasificacion', escalado=False)
    filas_clf.append((f'Walk-forward fold {k}', *_clf_minimo(a, b, c, d)))
tabla_humo_clf = pd.DataFrame(filas_clf, columns=['evaluación', 'AUC', 'F1 (umbral 0.5)', 'F1 trivial (siempre "habrá evento")', '% con evento (val.)']).round(3)
display(tabla_humo_clf)

Xr_tr, yr_tr = obtener_datos('entrenamiento', 'regresion', escalado=False)
Xr_va, yr_va = obtener_datos('validacion',    'regresion', escalado=False)
filas_reg = []
for t in TARGETS_REG:
    reg = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=SEMILLA).fit(Xr_tr, yr_tr[t])
    filas_reg.append({'regresor (salida)': t, 'MAE modelo mínimo (h)': round(mean_absolute_error(yr_va[t], reg.predict(Xr_va)), 3),
                      'MAE trivial: media de entrenamiento (h)': round(mean_absolute_error(yr_va[t], np.full(len(yr_va), yr_tr[t].mean())), 3)})
display(pd.DataFrame(filas_reg))
print(f'Matrices: clasificador {Xc_tr.shape} / {Xc_va.shape}  ·  regresor {Xr_tr.shape} / {Xr_va.shape}  ·  NaN en X: {int(Xc_tr.isna().sum().sum() + Xr_tr.isna().sum().sum())}')

auc_medio = tabla_humo_clf.loc[tabla_humo_clf['evaluación'].str.startswith('Walk'), 'AUC'].mean()
if auc_medio < 0.55:
    print(f'\nAVISO: AUC medio en walk-forward = {auc_medio:.2f} (azar = 0.50). Con las variables actuales, el modelo mínimo NO anticipa el evento '
          'de forma estable entre periodos. No es un error de construcción (las verificaciones de causalidad y de cálculo manual pasan): '
          'es un hallazgo sobre los datos/etiqueta que conviene revisar con la asesora antes de HU07 (ver decisiones abiertas).')
else:
    print(f'\nAUC medio en walk-forward = {auc_medio:.2f}: hay señal por encima del azar; HU07/HU08 la cuantificarán con el benchmark completo.')

### Exportación de entregables de HU06

| Archivo | Contenido |
|---|---|
| `dataset_features_HU06.csv` | Predictores **normalizados** + columnas de control (`particion`, `entrenable_clasificador`, `entrenable_regresores`, proporción imputada) + los 3 *targets* |
| `dataset_features_HU06_sin_escalar.csv` | Lo mismo en unidades originales (modelos de árboles, interpretación con SHAP) |
| `escalador_HU06.joblib` | Escalador ajustado solo con entrenamiento + lista de columnas (para reutilizarlo en HU13 al hacer predicciones nuevas) |
| `catalogo_variables_HU06.csv` | Una fila por variable: variable base, familia, parámetro en horas, si se normaliza |
| `folds_walk_forward_HU06.csv` | Fechas de cada ventana walk-forward |
| `config_features_HU06.json` | Parámetros, cortes de partición y versiones de librerías para reproducir el proceso |

In [ ]:
# ==== Exportación de entregables de HU06 ====
meta_cols = pd.DataFrame({
    'particion': particion, 'apto_modelado': df_hu05['apto_modelado'], 'racha_apto_h': racha_apto,
    'entrenable_clasificador': entrenable_clf.astype(int), 'entrenable_regresores': entrenable_reg.astype(int),
})
meta_cols = meta_cols.join(meta_calidad[[c for c in meta_calidad.columns if c not in X_COLS]])   # proporción imputada (si no es predictor)
dataset_hu06_esc = pd.concat([meta_cols, X_esc[X_COLS], df_hu05[TARGETS]], axis=1)
dataset_hu06_raw = pd.concat([meta_cols, X_modelo[X_COLS], df_hu05[TARGETS]], axis=1)

dataset_hu06_esc.to_csv('dataset_features_HU06.csv', float_format='%.8g')
dataset_hu06_raw.to_csv('dataset_features_HU06_sin_escalar.csv', float_format='%.8g')
joblib.dump({'escalador': escalador, 'tipo': ESCALADOR, 'columnas_escaladas': COLS_ESCALAR, 'columnas_modelo': X_COLS}, 'escalador_HU06.joblib')
catalogo.assign(en_X=catalogo['variable'].isin(X_COLS)).to_csv('catalogo_variables_HU06.csv', index=False)
pd.DataFrame(FOLDS).to_csv('folds_walk_forward_HU06.csv', index=False)

config_hu06 = {
    'insumo': RUTA_DATASET, 'horizonte_h': HORIZONTE_H, 'semilla': SEMILLA,
    'rezagos_h': LAGS_H, 'ventanas_h': VENTANAS_H, 'tasas_cambio_h': TASAS_CAMBIO_H,
    'horas_aptas_consecutivas_minimas': MIN_RACHA_H, 'max_frac_imputada_ventana': MAX_FRAC_IMPUTADA_VENTANA,
    'frac_imputada_como_predictor': INCLUIR_FRAC_IMPUTADA_EN_X,
    'columnas_descartadas': COLUMNAS_DESCARTADAS, 'escalador': ESCALADOR,
    'particion': {'frac_train': FRAC_TRAIN, 'frac_val': FRAC_VAL, 'purga_h': GAP_H,
                  'corte_entrenamiento': str(corte_train), 'corte_validacion': str(corte_val)},
    'n_folds_walk_forward': N_FOLDS_WF, 'n_predictores': len(X_COLS), 'predictores': X_COLS,
    'targets': {'clasificador': TARGET_CLF, 'regresor_doble_salida': TARGETS_REG},
    'versiones': {'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__},
}
Path('config_features_HU06.json').write_text(json.dumps(config_hu06, indent=2, ensure_ascii=False), encoding='utf-8')

archivos_hu06 = ['dataset_features_HU06.csv', 'dataset_features_HU06_sin_escalar.csv', 'escalador_HU06.joblib',
                 'catalogo_variables_HU06.csv', 'folds_walk_forward_HU06.csv', 'config_features_HU06.json']
print('Archivos generados:')
for f in archivos_hu06:
    print(f'  - {f}  ({Path(f).stat().st_size/1024:,.0f} KB)')

# Comprobación de ida y vuelta: el CSV exportado contiene exactamente lo que usa el modelo
chk = pd.read_csv('dataset_features_HU06.csv', index_col='fecha_hora', parse_dates=['fecha_hora'])
assert chk.shape == dataset_hu06_esc.shape and np.allclose(chk[X_COLS].fillna(0), dataset_hu06_esc[X_COLS].fillna(0), atol=1e-6)
print(f'Ida y vuelta del CSV: OK  ({chk.shape[0]:,} filas x {chk.shape[1]} columnas)')

try:
    from google.colab import files
    # for f in archivos_hu06: files.download(f)       # descomentar para descargar
except ImportError:
    pass

### Cómo cargar el dataset en HU07 (ejemplo mínimo)

```python
# Clasificador de ocurrencia (F1-score y AUC)
X_tr, y_tr = obtener_datos('entrenamiento', 'clasificacion')
X_va, y_va = obtener_datos('validacion',    'clasificacion')

# Regresor de doble salida (MAE, RMSE, MAPE, R²): y tiene 2 columnas
Xr_tr, yr_tr = obtener_datos('entrenamiento', 'regresion')     # [horas_hasta_inicio_24h, duracion_indisponibilidad_h]

# Estabilidad entre periodos (walk-forward): el escalador se reajusta dentro de cada fold
for k in range(1, N_FOLDS_WF + 1):
    X_tr, y_tr, X_va, y_va = obtener_fold(k, 'clasificacion')
```

Si HU07 se ejecuta en otra sesión, basta con leer `dataset_features_HU06.csv` y filtrar por `particion` y `entrenable_clasificador` / `entrenable_regresores`.

In [ ]:
# ==== HU06 - Definition of Done (calculada) ====
def estado(c): return 'CUMPLE' if c else 'NO CUMPLE'
dod_hu06 = pd.DataFrame([
    ['T11', 'Columna duplicada (velocidad en mph) y variables de diagnóstico del datalogger excluidas de los predictores', estado(chequeos['Entradas: columnas descartadas ausentes de X'])],
    ['T11', 'Dirección del viento codificada como seno y coseno', estado(chequeos['Codificación: sin² + cos² = 1 en la dirección del viento'])],
    ['T11', 'Partición temporal entrenamiento / validación / prueba con purga, documentada', estado(chequeos[f'Partición: purga de {GAP_H} h entre bloques'])],
    ['T11', 'Normalización ajustada solo con entrenamiento y aplicada a validación y prueba', estado(chequeos['Escalador: ajustado exactamente con las filas de entrenamiento'])],
    ['T12', 'Rezagos t-1, 2, 3, 6, 12 y 24 generados', estado(chequeos['Rezagos: se generan los 6 rezagos exigidos por el TB2 para cada variable base'])],
    ['T12', 'Ventanas móviles de 6 y 24 h (media y desviación) generadas', estado(chequeos['Ventanas: media (y desviación en continuas) de 6 y 24 h'])],
    ['T12', 'Ninguna variable derivada cruza un bloque no apto (apto_modelado)', estado(chequeos['Continuidad: ninguna fila entrenable cruza una hora no apta'])],
    ['T12', 'Proporción imputada por ventana calculada y umbral aplicado', estado(frac_max_col in meta_calidad.columns)],
    ['T12', 'Sin fuga de datos: variables solo con información de t hacia atrás', estado(chequeos['Causalidad: alterar el futuro NO cambia ninguna fila anterior (sin fuga)'])],
    ['-', 'Dataset reproducible (configuración, catálogo y escalador versionables)', estado(chequeos['Reproducibilidad: reconstruir las variables da exactamente el mismo resultado'] and all(Path(f).exists() for f in archivos_hu06))],
    ['-', 'Dataset apto para entrenamiento: dos tablas (clasificador y regresor de doble salida) sin NaN', estado(chequeos['Continuidad: filas entrenables sin NaN en X'])],
], columns=['Tarea', 'Criterio de aceptación', 'Estado'])
display(dod_hu06)

### Decisiones abiertas de HU06 para validar con la asesora / Product Owner

1. **Partición 70 / 15 / 15 con purga de 24 h** y **4 ventanas walk-forward**: son valores de partida. Si la asesora prefiere otro reparto, se cambia `FRAC_TRAIN`, `FRAC_VAL`, `GAP_H` o `N_FOLDS_WF` y se vuelve a ejecutar.
2. **Umbral de proporción imputada** (`MAX_FRAC_IMPUTADA_VENTANA = 50 %`): con los límites de interpolación de HU05 ya validados el efecto es mínimo (ver el recuento en T12), pero conviene confirmarlo.
3. **Rigor de la continuidad**: se exige que **todas** las horas entre `t-24` y `t` sean aptas, no solo la hora `t-24`. Es la lectura estricta de "una ventana que cruza un bloque no apto no es dato continuo"; cuesta unas 25 h de filas tras cada bloque.
4. **Escalador `standard` (z-score)**: razonable para el modelo lineal del *benchmark*; los modelos de árboles no lo requieren. Es parametrizable (`ESCALADOR`).
5. **Proporción imputada fuera de `X`** (`INCLUIR_FRAC_IMPUTADA_EN_X = False`): se usa como control de calidad, no como predictor, para que el modelo no aprenda patrones del mantenimiento de sensores.
6. **Prueba final intocable**: la partición `prueba` solo se usa en HU09.
7. **Señal predictiva de las variables actuales** (diagnóstico y prueba de humo de arriba): si el modelo mínimo no supera el azar de forma estable entre periodos, hay que decidir con la asesora si se incorporan variables adicionales (p. ej. periodo de ola, corrientes, nivel del mar, estado operativo previo), se revisa la definición de la etiqueta de 24 h o se ajusta la expectativa de las metas (R² ≥ 0.75, F1 ≥ 0.65, AUC ≥ 0.75) antes de entrenar HU07.

### Puente hacia HU07 (Entrenamiento del modelo)

HU07 parte de `obtener_datos(...)` / `obtener_fold(...)` o de `dataset_features_HU06.csv`. El **clasificador** (XGBoost y Random Forest Classifier, con F1-score y AUC) usa todas las filas con `entrenable_clasificador = 1`. El **regresor de doble salida** (baselines + RF, GBM, XGBoost, LightGBM; MAE, RMSE, MAPE y R²) usa solo `entrenable_regresores = 1`. El escalador debe reajustarse dentro de cada *fold*, tal como hace `obtener_fold`.